# Conditional Molecular Generation — DiT (QED-only, full logging)

Clean **CondDiT** (DiT backbone + adaLN-Zero), conditioned on **QED alone** for strong native steering. Everything saves to Drive: per-epoch checkpoints, history, generated molecules, QC report. **Resumes from any checkpoint**, logs to **Weights & Biases**.

**Run order:** cells 1–14 (data) → 15 (setup) → 16 (train, or skip & load) → 17 (sampler) → 18 (load best) → 19 (steering eval) → 20 (best-of-N) → 21 (QC).

To **resume**: set `RESUME_FROM='latest'` (default) or a path in the train cell. To **eval only**: run 1–15, 17, then 18 (`load_checkpoint('best')`), then 19–21.

## 1–6 · Data pipeline (reused, proven)
<sub>copied from organized cells 1–13: setup, config, features+CSV, split, tokenizer, dataset/loaders, `build_conditioning_for_qed`, `norm_stats`</sub>

In [ ]:
# CELL 1
!pip install rdkit selfies

In [ ]:
# CELL 2
# === Google Colab Setup ===
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# CELL 3
# Installation & Imports
import sys
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import json
from pathlib import Path
from tqdm.auto import tqdm
import math
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Optional

# Chemistry
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, QED, AllChem
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import BRICS, Recap, Fragments
from rdkit.DataStructs import TanimotoSimilarity

# SELFIES
import selfies as sf

# Plotting
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")
print(f"🔢 PyTorch version: {torch.__version__}")

In [ ]:
# CELL 4
from dataclasses import dataclass, field
from typing import List, Dict, Optional
import os

@dataclass
class D3PMConfig:
    """
    Centralized configuration to eliminate hardcoded values.
    
    CRITICAL: This is the single source of truth. All model components
    derive their sizes/parameters from this config to prevent mismatches.
    """
    
    # === ARCHITECTURE ===
    d_model: int = 512
    n_layers: int = 8
    n_heads: int = 8
    dropout: float = 0.1
    max_len: int = 128  # Main sequence length
    scaffold_max_len: int = 32  # Scaffold tokenization length
    
    # === DIFFUSION ===
    T: int = 100  # Diffusion timesteps
    beta_min: float = 0.0001
    beta_max: float = 0.02
    schedule_type: str = "cosine"  # "linear", "cosine", "sqrt"
    # FIX (C1): MUST use a schedule whose alpha_bar[-1] -> 0 so the forward process
    # fully converges to the all-[MASK] absorbing state. Reverse sampling STARTS from
    # all-[MASK]; if training never reaches ~100pct masked, the model generates from a
    # distribution it never saw. linear beta=[1e-4,0.02] leaves alpha_bar[-1]=0.364
    # (~36pct of tokens never masked) -> garbage generation regardless of train loss.
    # cosine verified to give alpha_bar[-1]=0.000000. Checked by validate_diffusion_schedule().
    maskgit_schedule: str = "cosine"  # "cosine", "linear", "root" — MaskGIT reveal schedule
    
    # === PROPERTIES ===
    # THIS IS THE SINGLE SOURCE OF TRUTH FOR PROPERTIES
    property_list: List[str] = field(default_factory=lambda: [
        'qed', 'mw', 'logp', 'hbd', 'hba', 'psa',
        'rotatable_bonds', 'aromatic_rings', 'num_rings',
        'num_heteroatoms', 'sp3_fraction'
    ])
    
    # Derived automatically
    @property
    def n_properties(self) -> int:
        return len(self.property_list)
    
    # === TRAINING THRESHOLDS (previously hardcoded) ===
    property_loss_t_threshold: int = 60  # STEP B: extend property pressure to high-noise
    #   steps (was 20). Structure is decided at high t during reverse sampling, so QED
    #   supervision must reach those steps, not just the near-clean t<20 region.
    diversity_t_threshold: int = 50  # Only apply diversity when t > this
    monotonicity_batch_interval: int = 10  # Run monotonicity every N batches
    
    # Entropy/diversity targets
    # ln(137) ≈ 4.92 nats = uniform; 4.0 activates penalty when predictions become peaked
    entropy_min_target: float = 4.0  # nats (was 2.0 — too low, penalty was always 0)
    attention_entropy_target: float = 1.0  # nats
    anti_collapse_cosine_threshold: float = 0.5  # Penalize if similarity > this
    
    # Per-property encoding schema — determines how each prop is fed to the model.
    # 'binned'     → needs extractor.property_bins entry + discrete bin embedding.
    # 'integer'    → count feature, encoded as normalized raw float (no bins).
    # 'continuous' → [0,1] fraction, encoded as raw float (no bins).
    # If you add a property to property_list, add it here too.
    PROPERTY_SCHEMA: Dict[str, str] = field(default_factory=lambda: {
        'qed':             'binned',      # [0, 1] discretized, ~5 bins
        'mw':              'binned',      # Molecular weight, ~6 bins
        'logp':            'binned',      # LogP, ~6 bins
        'hbd':             'binned',      # H-bond donors, ~5 bins
        'hba':             'binned',      # H-bond acceptors, ~6 bins
        'psa':             'binned',      # Polar surface area, ~5 bins
        'rotatable_bonds': 'integer',     # Count [0, ~15], no bins, raw value
        'aromatic_rings':  'integer',     # Count [0, ~5], no bins, raw value
        'num_rings':       'integer',     # Count [0, ~6], no bins, raw value
        'num_heteroatoms': 'integer',     # Count [0, ~10], no bins, raw value
        'sp3_fraction':    'continuous',  # Fraction [0, 1], no bins, raw value
    })

    # === FRAGMENTS ===
    n_fragment_bits: int = 256  # Must match MolecularFeatureExtractor.N_FRAGMENT_BITS
    
    # === SCAFFOLDS ===
    scaffold_fallback_smiles: Optional[str] = None  # Don't default to benzene!
    log_scaffold_failures: bool = True
    
    # === PATHS (environment-aware) ===
    def get_base_dir(self) -> Path:
        """Auto-detect if running in Colab or local."""
        if os.path.exists('/content/drive/MyDrive'):
            return Path('/content/drive/MyDrive')
        else:
            # Local fallback - use current notebook directory
            return Path(__file__).parent if '__file__' in globals() else Path.cwd()
    
    @property
    def data_dir(self) -> Path:
        base = self.get_base_dir()
        if 'MyDrive' in str(base):
            return base / 'Datasets' / 'Drug Discovery'
        else:
            return base / 'chembl_curated_out'
    
    @property
    def predictor_dir(self) -> Path:
        """Where the FROZEN property predictor checkpoint lives. Never write D3PM artifacts here."""
        base = self.get_base_dir()
        if 'MyDrive' in str(base):
            return base / 'Models' / 'D3PM_Ultimate'
        else:
            return base / 'models' / 'd3pm_runs'

    @property
    def run_dir(self) -> Path:
        """Where D3PM training checkpoints and logs go. Safe to write here."""
        base = self.get_base_dir()
        if 'MyDrive' in str(base):
            return base / 'Models' / 'D3PM_Ultimate'
        else:
            return base / 'models' / 'd3pm_runs'

    @property
    def model_dir(self) -> Path:
        """Alias for run_dir -- backward compatibility only. Prefer run_dir."""
        return self.run_dir
    
    # === SAMPLING DEFAULTS ===
    default_cfg_scale: float = 7.5
    default_temperature: float = 1.0
    
    # === VALIDATION ===
    validate_on_load: bool = True  # Run sanity checks when loading
    
    def __post_init__(self):
        """Ensure run directory exists. Never auto-creates predictor_dir (read-only)."""
        print(f"cwd: {Path.cwd()}")
        self.run_dir.mkdir(parents=True, exist_ok=True)
        if self.validate_on_load:
            print("Config initialized:")
            print(f"   Properties: {self.n_properties} ({', '.join(self.property_list[:3])}...)")
            print(f"   Data dir:      {self.data_dir}")
            print(f"   Predictor dir: {self.predictor_dir}  (read-only, frozen checkpoint)")
            print(f"   Run dir:       {self.run_dir}  (D3PM checkpoints + logs)")
            print(f"   Scaffold fallback: {'<from data>' if self.scaffold_fallback_smiles is None else self.scaffold_fallback_smiles}")
    
    def summary(self):
        """Print full config."""
        print("=" * 70)
        print("D3PM CONFIGURATION SUMMARY")
        print("=" * 70)
        print(f"Architecture: {self.n_layers} layers, {self.d_model} d_model, {self.n_heads} heads")
        print(f"Diffusion: T={self.T}, beta=[{self.beta_min}, {self.beta_max}], schedule={self.schedule_type}")
        print(f"Properties: {self.n_properties} - {self.property_list}")
        print(f"Max lengths: sequence={self.max_len}, scaffold={self.scaffold_max_len}")
        print(f"Training thresholds: prop_t<{self.property_loss_t_threshold}, div_t>{self.diversity_t_threshold}")
        print(f"Paths: data={self.data_dir}")
        print(f"       predictor_dir={self.predictor_dir}  (frozen, read-only)")
        print(f"       run_dir={self.run_dir}  (checkpoints + logs)")
        print("=" * 70)

# Initialize global config
config = D3PMConfig()
config.summary()

# Global assertions to catch configuration errors early

def validate_config_against_extractor(cfg: D3PMConfig, extractor):
    """
    Critical validation: ensure config matches data source.
    Call this AFTER extractor is created and BEFORE model init.

    Validates:
    - property_list ORDER matches between cfg and extractor
    - All 'binned' properties have matching bin definitions in extractor
    - 'integer'/'continuous' properties are consistently NOT binned
    - Fragment bit count matches
    - BRICS settings match class constants
    """
    print("\\nValidating configuration against extractor...")
    errors = []

    # 1. Property ORDER alignment -- silently wrong gradients if mismatched
    if hasattr(extractor, 'property_list'):
        if extractor.property_list != cfg.property_list:
            errors.append(
                f"PROPERTY ORDER MISMATCH!\\n"
                f"   Config order:    {cfg.property_list}\\n"
                f"   Extractor order: {extractor.property_list}\\n"
                f"   Dataset, predictor, and model will all compute different properties!"
            )
        else:
            print(f"  Property order aligned ({cfg.n_properties} props)")
    else:
        print("  Warning: extractor has no property_list attr -- add one and assert order")

    # 2. Validate each property against its schema
    schema = cfg.PROPERTY_SCHEMA
    for prop in cfg.property_list:
        encoding = schema.get(prop, 'UNKNOWN')
        if encoding == 'UNKNOWN':
            errors.append(f"'{prop}' is in property_list but missing from PROPERTY_SCHEMA!")
        elif encoding == 'binned':
            if prop not in extractor.property_bins:
                errors.append(f"'{prop}' is marked 'binned' but has no entry in extractor.property_bins!")
            else:
                n_bins = len(extractor.property_bins[prop]) - 1
                print(f"  {prop:20s}  binned       ({n_bins} bins)")
        elif encoding in ('integer', 'continuous'):
            if prop in extractor.property_bins:
                print(f"  Warning: {prop:20s}  {encoding} but has bins -- bins will be ignored")
            else:
                print(f"  {prop:20s}  {encoding:10s} (raw value, no bins)")

    # 3. Fragment bit count
    if extractor.N_FRAGMENT_BITS != cfg.n_fragment_bits:
        errors.append(
            f"Fragment bit mismatch: "
            f"config.n_fragment_bits={cfg.n_fragment_bits}, "
            f"extractor.N_FRAGMENT_BITS={extractor.N_FRAGMENT_BITS}"
        )
    else:
        print(f"  Fragment bits: {cfg.n_fragment_bits}")

    # 4. BRICS settings (dataset-build-time vs runtime must be identical)
    if hasattr(extractor, 'BRICS_MIN_FRAGMENT_SIZE'):
        if extractor.BRICS_MIN_FRAGMENT_SIZE != 3:
            errors.append(
                f"BRICS_MIN_FRAGMENT_SIZE={extractor.BRICS_MIN_FRAGMENT_SIZE} "
                f"(expected 3) -- fragment bits from dataset won't match generation!"
            )
        else:
            print(f"  BRICS minFragmentSize: {extractor.BRICS_MIN_FRAGMENT_SIZE}")
        if not extractor.CANONICALIZE_SMILES:
            errors.append("CANONICALIZE_SMILES=False -- fragment SMILES hashing will be inconsistent!")

    # Report
    if errors:
        print("\\nVALIDATION ERRORS:")
        for e in errors:
            print(f"  {e}")
        raise ValueError("Configuration validation failed -- fix before training!")

    print("Config validation passed!")
    return True


def validate_diffusion_schedule(model):
    """
    Validate diffusion schedule for DISCRETE absorbing-state diffusion.

    Cosine betas from continuous DDPM can exceed 1.0 for discrete diffusion,
    which makes the masking probability invalid and produces garbage samples
    without crashing.

    Checks:
    - 0 < beta_t < 1 for all t
    - alpha_bar strictly decreasing (more masking over time)
    - No NaNs / Infs
    - Boundary conditions: alpha_bar[0] close to 1, alpha_bar[-1] close to 0

    Call this after model initialization.
    """
    print("\\nValidating diffusion schedule...")
    errors = []

    betas = model.betas.cpu().float()
    alpha_bars = model.alpha_bars.cpu().float()

    # Check beta range
    if (betas <= 0).any():
        errors.append(f"betas has values <= 0: min={betas.min():.6f}")
    if (betas >= 1).any():
        bad_count = (betas >= 1).sum().item()
        errors.append(f"betas has {bad_count} values >= 1 (max={betas.max():.6f}) -- invalid masking probability!")
    if torch.isnan(betas).any() or torch.isinf(betas).any():
        errors.append("betas contains NaN or Inf!")

    # Check alpha_bar monotonically decreasing
    diffs = alpha_bars[1:] - alpha_bars[:-1]
    if (diffs >= 0).any():
        n_bad = (diffs >= 0).sum().item()
        errors.append(f"alpha_bar is NOT strictly decreasing at {n_bad} steps -- diffusion won't corrupt correctly!")
    if torch.isnan(alpha_bars).any() or torch.isinf(alpha_bars).any():
        errors.append("alpha_bars contains NaN or Inf!")

    # Boundary conditions
    if alpha_bars[0].item() < 0.9:
        errors.append(f"alpha_bar[0]={alpha_bars[0].item():.4f} < 0.9 -- too much corruption at t=0!")
    if alpha_bars[-1].item() > 0.1:
        errors.append(f"alpha_bar[T-1]={alpha_bars[-1].item():.4f} > 0.1 -- not enough corruption by t=T!")

    if errors:
        print("DIFFUSION SCHEDULE ERRORS:")
        for e in errors:
            print(f"  {e}")
        raise ValueError("Diffusion schedule validation failed!")

    print(f"  betas in (0,1): [{betas.min():.5f}, {betas.max():.5f}]")
    print(f"  alpha_bar: {alpha_bars[0]:.4f} -> {alpha_bars[-1]:.5f}, strictly decreasing")
    print(f"  No NaNs or Infs")
    print("Diffusion schedule validated!")
    return True

In [ ]:
# CELL 5
# ============================================================================
# UTILITY FUNCTIONS: Safe Scaffold Handling &  Validation
# ============================================================================

from collections import Counter

# Global counters for diagnostics
scaffold_failures = Counter()
scaffold_truncations = Counter()

def get_scaffold_smiles_safe(extractor, scaffold_id, fallback_strategy='most_common', verbose=False):
    """
    Safe scaffold lookup with fallback strategies.
    
    CRITICAL: Eliminates hardcoded benzene fallback that biases generation.
    
    Args:
        extractor: MolecularFeatureExtractor instance
        scaffold_id: Scaffold ID to look up
        fallback_strategy:
            - 'most_common': Use most frequent scaffold from training (recommended)
            - 'none': Return empty string (no scaffold conditioning)
            - str: Use specific SMILES as fallback
        verbose: Whether to log scaffold failures
    
    Returns:
        scaffold_smiles: str
    """
    if extractor is None or not hasattr(extractor, 'id_to_scaffold'):
        return '' if fallback_strategy in ('most_common', 'none') else fallback_strategy

    smiles = extractor.id_to_scaffold.get(scaffold_id)
    
    if smiles is None:
        scaffold_failures[scaffold_id] += 1
        
        if verbose and scaffold_failures[scaffold_id] == 1:
            print(f"⚠️ Scaffold {scaffold_id} not found (total failures: {sum(scaffold_failures.values())})")
        
        if fallback_strategy == 'most_common':
            # Use most common scaffold from training data
            if hasattr(extractor, 'scaffold_to_id') and len(extractor.scaffold_to_id) > 0:
                # Find most common scaffold by counting id_to_scaffold values
                scaffold_counts = Counter(extractor.id_to_scaffold.values())
                smiles = scaffold_counts.most_common(1)[0][0]
            else:
                smiles = ''  # No scaffolds available
        elif fallback_strategy == 'none':
            smiles = ''
        else:
            smiles = fallback_strategy  # User-specified SMILES
    
    return smiles


def validate_model_config(model, extractor, config):
    """
    Critical validation: catch configuration mismatches before training.
    
    Call this after model initialization to prevent silent failures.
    """
    print("\n" + "="*70)
    print("🔍 CONFIGURATION VALIDATION")
    print("="*70)
    
    errors = []
    warnings = []
    
    # 1. Check bin sizes match extractor
    if hasattr(model, 'bin_sizes'):
        for prop in ['qed', 'mw', 'logp', 'hbd', 'hba', 'psa']:
            if prop in extractor.property_bins:
                expected_bins = len(extractor.property_bins[prop])
                actual_bins = model.bin_sizes.get(prop, 0)
                
                if expected_bins != actual_bins:
                    errors.append(
                        f"❌ BIN MISMATCH: {prop} has {expected_bins} bins in data "
                        f"but {actual_bins} in model! This will cause IndexError!"
                    )
                else:
                    print(f"  ✅ {prop.upper()}: {actual_bins} bins (matches data)")
    else:
        warnings.append("⚠️ Model does not have bin_sizes attribute (using hardcoded?)")
    
    # 2. Check fragment bits
    model_frag_bits = model.fragment_encoder.in_features if hasattr(model, 'fragment_encoder') else None
    extractor_frag_bits = MolecularFeatureExtractor.N_FRAGMENT_BITS
    
    if model_frag_bits != extractor_frag_bits:
        errors.append(
            f"❌ FRAGMENT BITS MISMATCH: model={model_frag_bits}, "
            f"extractor={extractor_frag_bits}"
        )
    else:
        print(f"  ✅ Fragment bits: {model_frag_bits}")
    
    # 3. Check n_properties
    if hasattr(model, 'n_properties'):
        if model.n_properties != config.n_properties:
            errors.append(
                f"❌ PROPERTY COUNT MISMATCH: model={model.n_properties}, "
                f"config={config.n_properties}"
            )
        else:
            print(f"  ✅ Properties: {config.n_properties} ({config.property_list[0]}, ...)")
    
    # 4. Check diffusion timesteps
    if model.T != config.T:
        warnings.append(f"⚠️ Timestep mismatch: model.T={model.T}, config.T={config.T}")
    else:
        print(f"  ✅ Diffusion steps: {config.T}")
    
    # 5. Check max_len
    if model.max_len != config.max_len:
        warnings.append(f"⚠️ Max length mismatch: model={model.max_len}, config={config.max_len}")
    else:
        print(f"  ✅ Sequence length: {config.max_len}")
    
    # 6. Check scaffold embedding size
    n_scaffolds = len(extractor.scaffold_to_id)
    if hasattr(model, 'scaffold_embed'):
        scaffold_embed_size = model.scaffold_embed.num_embeddings
        if scaffold_embed_size < n_scaffolds:
            errors.append(
                f"❌ SCAFFOLD EMBEDDING TOO SMALL: {scaffold_embed_size} embeddings "
                f"but {n_scaffolds} scaffolds in data!"
            )
        else:
            print(f"  ✅ Scaffold embeddings: {scaffold_embed_size} (data has {n_scaffolds})")
    
    print("="*70)
    
    # Print warnings
    if warnings:
        print("\n⚠️ WARNINGS:")
        for w in warnings:
            print(f"  {w}")
    
    # Print errors and fail if critical
    if errors:
        print("\n❌ CRITICAL ERRORS FOUND:")
        for e in errors:
            print(f"  {e}")
        print("\n💥 FIX THESE BEFORE TRAINING OR YOU WILL GET CRASHES!\n")
        raise ValueError("Configuration validation failed - see errors above")
    
    print("\n✅ ALL CRITICAL VALIDATIONS PASSED!\n")
    return True


def log_scaffold_diagnostics():
    """Print scaffold failure/truncation statistics."""
    if scaffold_failures:
        print(f"\n📊 Scaffold Diagnostics:")
        print(f"   Unique failed scaffolds: {len(scaffold_failures)}")
        print(f"   Total failures: {sum(scaffold_failures.values())}")
        print(f"   Top 5 failed: {scaffold_failures.most_common(5)}")
    
    if scaffold_truncations:
        print(f"   Truncations: {sum(scaffold_truncations.values())}")
        print(f"   Top 5 truncated: {scaffold_truncations.most_common(5)}")


print("✅ Utility functions loaded: get_scaffold_smiles_safe(), validate_model_config()")

In [ ]:
# CELL 6
class MolecularFeatureExtractor:
    """
    Comprehensive feature extractor for molecular generation.
    Handles scaffolds, fragments, property bins, and QED decomposition.
    """
    # FIX 2: Class-level constants to prevent silent mismatch
    N_FRAGMENT_BITS = 256  # Must match config.n_fragment_bits

    # === FRAGMENT EXTRACTION SETTINGS ===
    # These MUST be identical between dataset-build-time (feature extraction)
    # and generation time. Any deviation makes fragment bit vectors meaningless.
    # Assert these when loading a pre-built dataset:
    #   assert MolecularFeatureExtractor.BRICS_MIN_FRAGMENT_SIZE == 3
    #   assert MolecularFeatureExtractor.CANONICALIZE_SMILES == True
    BRICS_MIN_FRAGMENT_SIZE: int = 3   # minFragmentSize arg to BRICSDecompose
    CANONICALIZE_SMILES: bool = True    # Canonicalize SMILES before fragment hashing

    # Literature-derived sensitivity priors for common edits
    SENSITIVITY_PRIORS = {
        'add_OH': {'mw': +17.0, 'logp': -0.67, 'hbd': +1, 'hba': +1, 'psa': +20.2},
        'add_CH3': {'mw': +14.0, 'logp': +0.56, 'hbd': 0, 'hba': 0, 'psa': 0.0},
        'add_F': {'mw': +18.0, 'logp': +0.14, 'hbd': 0, 'hba': +1, 'psa': 0.0},
        'add_Cl': {'mw': +34.5, 'logp': +0.71, 'hbd': 0, 'hba': 0, 'psa': 0.0},
        'add_NH2': {'mw': +16.0, 'logp': -1.0, 'hbd': +1, 'hba': +1, 'psa': +26.0},
        'ring_open': {'mw': 0, 'logp': -0.5, 'hbd': 0, 'hba': 0, 'psa': +10.0},
        'ring_close': {'mw': 0, 'logp': +0.5, 'hbd': 0, 'hba': 0, 'psa': -10.0},
    }
    
    def __init__(self, property_bins=None):
        """
        Args:
            property_bins: Optional dict of {property_name: [bin_edges]}
                          If None, uses default bins
        """
        # Scaffold tracking
        self.scaffold_to_id = {}
        self.id_to_scaffold = {}
        self.next_scaffold_id = 0
        
        # Fragment tracking
        self.fragment_to_id = {}
        self.id_to_fragment = {}
        self.fragment_counts = Counter()  # Track frequency for top-k selection
        
        # Property bin edges (data-driven or default)
        if property_bins is not None:
            self.property_bins = property_bins
        else:
            self.property_bins = {
                'qed': [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
                'mw': [0, 200, 300, 400, 500, 600, 800],
                'logp': [-3.0, 0.0, 1.0, 2.0, 3.0, 5.0, 7.0],
                'hbd': [0, 1, 2, 3, 5, 10],
                'hba': [0, 2, 4, 6, 8, 15],
                'psa': [0, 40, 80, 120, 160, 200]
            }
        
        # Sensitivity priors
        self.sensitivity_priors = self.SENSITIVITY_PRIORS
    
    def get_scaffold(self, mol):
        """Extract Murcko scaffold and assign/retrieve ID"""
        try:
            scaffold = MurckoScaffold.GetScaffoldForMol(mol)
            scaffold_smiles = Chem.MolToSmiles(scaffold)
        except:
            scaffold_smiles = ''
        
        if scaffold_smiles not in self.scaffold_to_id:
            self.scaffold_to_id[scaffold_smiles] = self.next_scaffold_id
            self.id_to_scaffold[self.next_scaffold_id] = scaffold_smiles
            self.next_scaffold_id += 1
        
        return self.scaffold_to_id[scaffold_smiles], scaffold_smiles
    
    def compute_qed_components(self, mol):
        """Get individual QED property components"""
        try:
            desc = Descriptors.MolWt(mol), Descriptors.MolLogP(mol)
            return {
                'mw': Descriptors.MolWt(mol),
                'logp': Descriptors.MolLogP(mol),
                'hbd': Descriptors.NumHDonors(mol),
                'hba': Descriptors.NumHAcceptors(mol),
                'psa': Descriptors.TPSA(mol),
                'rotb': Descriptors.NumRotatableBonds(mol),
                'arom': len(mol.GetAromaticAtoms()) / max(mol.GetNumAtoms(), 1),
                'alerts': 0,  # Structural alerts require pattern matching
            }
        except:
            return {k: 0.0 for k in ['mw', 'logp', 'hbd', 'hba', 'psa', 'rotb', 'arom', 'alerts']}
    
    def discretize_property(self, value, prop_name):
        """Convert continuous property to discrete bin index"""
        if prop_name not in self.property_bins:
            return 0, 'unknown'
        
        edges = self.property_bins[prop_name]
        bin_idx = np.searchsorted(edges[1:], value, side='right')
        bin_idx = min(bin_idx, len(edges) - 2)  # Clamp to valid range
        
        # Create label
        if bin_idx == 0:
            label = f'{prop_name}<{edges[1]}'
        elif bin_idx >= len(edges) - 2:
            label = f'{prop_name}>={edges[-2]}'
        else:
            label = f'{edges[bin_idx]}<={prop_name}<{edges[bin_idx+1]}'
        
        return int(bin_idx), label
    
    def get_fragments(self, mol, min_size=3, max_size=8):
        """Extract molecular fragments using BRICS decomposition"""
        try:
            # Use BRICS decomposition
            frags = BRICS.BRICSDecompose(mol, minFragmentSize=min_size)
            frag_list = list(frags)[:20]  # Limit fragments
            
            # Track fragment frequencies
            for frag in frag_list:
                self.fragment_counts[frag] += 1
            
            return frag_list
        except:
            return []
    
    def build_fragment_dictionary(self, top_k=256):
        """Build fragment dictionary from accumulated fragment counts.
        Call after processing all molecules."""
        most_common = self.fragment_counts.most_common(top_k)
        self.fragment_to_id = {frag: idx for idx, (frag, _) in enumerate(most_common)}
        self.id_to_fragment = {idx: frag for frag, idx in self.fragment_to_id.items()}
        print(f"📚 Built fragment dictionary: {len(self.fragment_to_id)} fragments from {sum(self.fragment_counts.values())} total occurrences")
    
    def get_fragment_bits(self, mol, n_bits=None):
        """Convert fragment presence to binary vector of size N_FRAGMENT_BITS"""
        if n_bits is None:
            n_bits = self.N_FRAGMENT_BITS
        
        bits = np.zeros(n_bits, dtype=np.float32)
        
        if not self.fragment_to_id:
            return bits
        
        try:
            frags = BRICS.BRICSDecompose(mol, minFragmentSize=3)
            for frag in frags:
                if frag in self.fragment_to_id:
                    idx = self.fragment_to_id[frag]
                    if idx < n_bits:
                        bits[idx] = 1.0
        except:
            pass
        
        return bits
    
    def compute_property_sensitivity(self, mol, qed):
        """
        Return the sensitivity priors table as a dict.
        These are literature-derived expected Δproperty values for common edits.
        Used as weak supervision to regularize condition embedding directions.
        """
        return self.sensitivity_priors
    
    def extract_all_features(self, smiles):
        """Extract complete feature set for a molecule"""
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        
        # 1. Compute QED and decompose into components
        qed = QED.qed(mol)
        components = self.compute_qed_components(mol)
        
        # 2. Get scaffold information
        scaffold_id, scaffold_smiles = self.get_scaffold(mol)
        
        # 3. Discretize key properties (6 properties)
        qed_bin, qed_label = self.discretize_property(qed, 'qed')
        mw_bin, _ = self.discretize_property(components['mw'], 'mw')
        logp_bin, _ = self.discretize_property(components['logp'], 'logp')
        hbd_bin, hbd_label = self.discretize_property(components['hbd'], 'hbd')
        hba_bin, hba_label = self.discretize_property(components['hba'], 'hba')
        psa_bin, psa_label = self.discretize_property(components['psa'], 'psa')
        
        # 4. Extract fragments (multi-scale)
        fragments = self.get_fragments(mol)
        
        # 5. Get fragment bit vector
        fragment_bits = self.get_fragment_bits(mol)
        
        # 6. Sensitivity priors
        sensitivity = self.compute_property_sensitivity(mol, qed)
        
        return {
            'canonical_smiles': Chem.MolToSmiles(mol),  # Canonical for uniqueness
            'smiles': smiles,  # Keep original for reference
            'qed': qed,
            'scaffold_id': scaffold_id,
            'scaffold_smiles': scaffold_smiles,
            'qed_bin': qed_bin,
            'mw_bin': mw_bin,
            'logp_bin': logp_bin,
            'hbd_bin': hbd_bin,
            'hba_bin': hba_bin,
            'psa_bin': psa_bin,
            'qed_label': qed_label,
            'fragments': fragments,
            'fragment_bits': fragment_bits,
            'sensitivity': sensitivity,
            **components  # Unpack all QED components
        }

print("✅ MolecularFeatureExtractor created")

In [ ]:
# CELL 7
# Load and process dataset with ALL features
# FIX 14: Add caching to avoid recomputation
# FIX D12: Standardize column names
print("📂 Loading dataset...")

df = pd.read_csv(config.data_dir / 'molecules_200k_selfies_clean.csv')
# FIX D12: Standardize column names for this dataset
rename_map = {}
if 'canonical_smiles' in df.columns and 'smiles' not in df.columns:
    rename_map['canonical_smiles'] = 'smiles'
elif 'SMILES' in df.columns and 'smiles' not in df.columns:
    rename_map['SMILES'] = 'smiles'
if 'alogp' in df.columns and 'logp' not in df.columns:
    rename_map['alogp'] = 'logp'
if rename_map:
    df = df.rename(columns=rename_map)
    print(f'   Renamed columns: {rename_map}')
print(f"📊 Loaded {len(df)} molecules")

# Create feature extractor
# FIX (T6): derive QUANTILE (equal-frequency) bin edges from the data so every bin is
# populated. Hardcoded MW bins left 2 of 7 bins EMPTY on this 200k set -> dead bin
# embeddings -> garbage conditioning there. Quantile bins guarantee ~equal mass/bin.
def _quantile_edges(series, n_bins):
    import numpy as _np
    qs = _np.linspace(0, 1, n_bins + 1)
    edges = _np.unique(_np.quantile(series.dropna().values, qs))
    edges[0] = edges[0] - 1e-6
    edges[-1] = edges[-1] + 1e-6
    return [float(e) for e in edges]
_bin_counts = {'qed': 5, 'mw': 6, 'logp': 6, 'hbd': 5, 'hba': 5, 'psa': 5}
_data_bins = {}
for _p, _nbins in _bin_counts.items():
    _col = _p if _p in df.columns else ('logP' if _p == 'logp' and 'logP' in df.columns else None)
    if _col is not None:
        _data_bins[_p] = _quantile_edges(df[_col], _nbins)
_DATA_BINS_SIGNATURE = json.dumps(_data_bins, sort_keys=True)
print('   Quantile bins/prop: ' + str({k: len(v)-1 for k, v in _data_bins.items()}))
extractor = MolecularFeatureExtractor(property_bins=_data_bins)

# Check if cached features exist
cache_file = str(config.data_dir / 'molecules_200k_enriched_features.parquet')

# FIX (T5): only use the cache if BOTH the parquet AND the state JSON exist (no
# FileNotFoundError if the JSON is missing). FIX (T6): and only if the cached bin
# signature matches the freshly-derived quantile bins -- otherwise the cached *_bin
# columns were computed under different edges and would mis-index the bin embeddings,
# so we rebuild. The rebuild path (else:) regenerates and re-caches with the new bins.
_state_path = config.model_dir / 'feature_extractor_state.json'
_use_cache = Path(cache_file).exists() and _state_path.exists()
if _use_cache:
    with open(_state_path, 'r') as _f:
        _cached_sig = json.dumps(json.load(_f).get('property_bins', {}), sort_keys=True)
    if _cached_sig != _DATA_BINS_SIGNATURE:
        print('   [T6] cached bins differ from data-derived quantile bins -> rebuilding.')
        _use_cache = False

if _use_cache:
    print("\n✅ Loading cached enriched features...")
    df_enriched = pd.read_parquet(cache_file)
    
    # Restore extractor state
    with open(_state_path, 'r') as f:
        state = json.load(f)
        extractor.scaffold_to_id = {k: int(v) for k, v in state['scaffold_to_id'].items()}
        extractor.id_to_scaffold = {int(k): v for k, v in state['id_to_scaffold'].items()}
        extractor.next_scaffold_id = state['next_scaffold_id']
        extractor.property_bins = state['property_bins']
        if 'fragment_to_id' in state:
            extractor.fragment_to_id = state['fragment_to_id']
            extractor.id_to_fragment = {int(k): v for k, v in state['id_to_fragment'].items()}
    
    print(f"📊 Loaded {len(df_enriched)} molecules from cache")
else:
    # FIX 14: Extract features sequentially with progress bar
    print("\n🔬 Extracting advanced features (this may take a while)...")
    
    enriched_data = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting features"):
        # FIX D12: Use standardized 'smiles' column
        features = extractor.extract_all_features(row['smiles'])
        if features is not None:
            enriched_data.append(features)
    
    # Create enriched DataFrame
    df_enriched = pd.DataFrame(enriched_data)
    
    # Build fragment dictionary from accumulated fragment counts
    extractor.build_fragment_dictionary(top_k=MolecularFeatureExtractor.N_FRAGMENT_BITS)
    
    # Recompute fragment_bits using the built dictionary
    print("🔗 Computing fragment bit vectors...")
    def _compute_frag_bits(smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.zeros(MolecularFeatureExtractor.N_FRAGMENT_BITS, dtype=np.float32).tolist()
        return extractor.get_fragment_bits(mol).tolist()
    
    df_enriched['fragment_bits'] = df_enriched['canonical_smiles'].apply(_compute_frag_bits)
    print(f"✅ Fragment bits computed for {len(df_enriched)} molecules")
    
    # Save to cache
    Path(cache_file).parent.mkdir(parents=True, exist_ok=True)
    df_enriched.to_parquet(cache_file, index=False)
    print(f"\n💾 Cached enriched features to {cache_file}")

from rdkit.Chem import rdMolDescriptors
# FIX: Standardize column names to match Dataset expectations
# MolecularFeatureExtractor uses abbreviated names; Dataset expects full names
if 'rotb' in df_enriched.columns and 'rotatable_bonds' not in df_enriched.columns:
    df_enriched = df_enriched.rename(columns={'rotb': 'rotatable_bonds'})
    print('   Renamed: rotb -> rotatable_bonds')
# Compute missing descriptors: aromatic_rings, num_rings, num_heteroatoms, sp3_fraction
if 'num_rings' not in df_enriched.columns:
    print('Computing missing descriptors (aromatic_rings, num_rings, num_heteroatoms, sp3_fraction)...')
    _mols = [Chem.MolFromSmiles(s) for s in df_enriched['smiles']]
    df_enriched['aromatic_rings'] = [rdMolDescriptors.CalcNumAromaticRings(m) if m else 0 for m in _mols]
    df_enriched['num_rings'] = [m.GetRingInfo().NumRings() if m else 0 for m in _mols]
    df_enriched['num_heteroatoms'] = [rdMolDescriptors.CalcNumHeteroatoms(m) if m else 0 for m in _mols]
    df_enriched['sp3_fraction'] = [float(rdMolDescriptors.CalcFractionCSP3(m)) if m else 0.0 for m in _mols]
    del _mols
    print('Missing descriptors computed')


# Save extractor state (including fragment dictionary)
extractor_state_file = str(config.model_dir / 'feature_extractor_state.json')
Path(extractor_state_file).parent.mkdir(parents=True, exist_ok=True)
with open(extractor_state_file, 'w') as f:
    json.dump({
        'scaffold_to_id': extractor.scaffold_to_id,
        'id_to_scaffold': extractor.id_to_scaffold,
        'next_scaffold_id': extractor.next_scaffold_id,
        'property_bins': extractor.property_bins,
        'fragment_to_id': extractor.fragment_to_id,
        'id_to_fragment': {str(k): v for k, v in extractor.id_to_fragment.items()},
    }, f)

print(f"\n✅ Processed {len(df_enriched)} molecules with enhanced features")
print(f"📊 Number of unique scaffolds: {len(extractor.scaffold_to_id)}")
print(f"\n📋 Feature columns: {list(df_enriched.columns)}")

# Display statistics
print("\n📈 Property distributions:")
for prop in ['qed', 'mw', 'logp', 'hbd', 'hba']:
    print(f"  {prop}: {df_enriched[prop].mean():.3f} ± {df_enriched[prop].std():.3f}")

print("\n🏷️  QED bin distribution:")
print(df_enriched['qed_label'].value_counts())

In [ ]:
# CELL 8
# Create property-stratified train/val split
print("🎯 Creating property-stratified split...")

# Stratify by QED bins to ensure balanced representation
from sklearn.model_selection import GroupShuffleSplit

# FIX (R2): SCAFFOLD-GROUPED split, not a random/stratified one. A random split puts
# molecules that share a Bemis-Murcko scaffold (near-duplicate analogs) into BOTH
# train and val, which makes val_loss optimistic and corrupts checkpoint selection
# (best_val_loss). Grouping whole scaffolds into train XOR val makes val measure true
# generalization to novel chemotypes. NOTE: val_loss will look WORSE than before --
# that is correct, not a regression. (R2 decision: scaffold split chosen.)
_gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
_tr_idx, _va_idx = next(_gss.split(df_enriched, groups=df_enriched['scaffold_id']))
train_df = df_enriched.iloc[_tr_idx].reset_index(drop=True)
val_df   = df_enriched.iloc[_va_idx].reset_index(drop=True)
_tr_scaf = set(train_df['scaffold_id']); _va_scaf = set(val_df['scaffold_id'])
assert _tr_scaf.isdisjoint(_va_scaf), "scaffold leakage between train and val!"
print(f"   Scaffold split: {len(_tr_scaf)} train scaffolds, {len(_va_scaf)} val scaffolds, "
      f"0 shared")

print(f"✅ Training set: {len(train_df)} molecules")
print(f"✅ Validation set: {len(val_df)} molecules")

# Verify stratification
print("\n📊 Training set QED distribution:")
print(train_df['qed_label'].value_counts(normalize=True))

print("\n📊 Validation set QED distribution:")
print(val_df['qed_label'].value_counts(normalize=True))

In [ ]:
# CELL 9
class MultiScaleTokenizer:
    """
    Tokenizer with multi-scale representations:
    - Token level (SELFIES)
    - Fragment level (BRICS decomposition)
    - Scaffold level (Bemis-Murcko)
    """
    
    def __init__(self, vocab_file=None):
        # Special tokens
        self.pad_token = '[PAD]'
        self.bos_token = '[BOS]'
        self.eos_token = '[EOS]'
        self.mask_token = '[MASK]'
        self.unk_token = '[UNK]'  # FIX 9: Add UNK token separate from MASK
        
        if vocab_file and Path(vocab_file).exists():
            with open(vocab_file, 'r') as f:
                self.token_to_id = json.load(f)
            self.id_to_token = {v: k for k, v in self.token_to_id.items()}
        else:
            # Will be built from data
            self.token_to_id = {
                self.pad_token: 0,
                self.bos_token: 1,
                self.eos_token: 2,
                self.mask_token: 3,
                self.unk_token: 4,  # FIX 9: UNK is distinct from MASK
            }
            self.id_to_token = {v: k for k, v in self.token_to_id.items()}
        
        self.vocab_size = len(self.token_to_id)
    
    def build_vocab(self, smiles_list, max_vocab_size=5000):
        """Build vocabulary from SELFIES tokens"""
        token_counts = Counter()
        
        for smiles in tqdm(smiles_list, desc="Building vocab"):
            try:
                selfies = sf.encoder(smiles)
                tokens = list(sf.split_selfies(selfies))
                token_counts.update(tokens)
            except:
                continue
        
        # Add most common tokens
        for token, _ in token_counts.most_common(max_vocab_size):
            if token not in self.token_to_id:
                self.token_to_id[token] = len(self.token_to_id)
                self.id_to_token[len(self.id_to_token)] = token
        
        self.vocab_size = len(self.token_to_id)
        print(f"✅ Vocabulary size: {self.vocab_size}")
    
    def encode(self, smiles, max_len=128):
        """Encode SMILES to token IDs"""
        try:
            selfies = sf.encoder(smiles)
            tokens = [self.bos_token] + list(sf.split_selfies(selfies)) + [self.eos_token]
            
            # Convert to IDs - FIX 9: Map unknown to UNK not MASK
            ids = [self.token_to_id.get(t, self.token_to_id[self.unk_token]) for t in tokens]
            
            # Truncate or pad
            if len(ids) > max_len:
                ids = ids[:max_len-1] + [self.token_to_id[self.eos_token]]
            else:
                ids = ids + [self.token_to_id[self.pad_token]] * (max_len - len(ids))
            
            return ids
        except:
            return [self.token_to_id[self.pad_token]] * max_len
    
    def decode(self, token_ids):
        """Decode token IDs to SMILES"""
        tokens = []
        for tid in token_ids:
            token = self.id_to_token.get(tid, self.unk_token)
            if token == self.eos_token:
                break
            if token not in [self.pad_token, self.bos_token, self.mask_token, self.unk_token]:
                tokens.append(token)
        
        try:
            selfies = ''.join(tokens)
            smiles = sf.decoder(selfies)
            return smiles
        except:
            return ""
    
    def save(self, vocab_file):
        """Save vocabulary"""
        with open(vocab_file, 'w') as f:
            json.dump(self.token_to_id, f)
        print(f"💾 Vocabulary saved to {vocab_file}")

print("✅ MultiScaleTokenizer created")

In [ ]:
# CELL 10
# Load or build vocabulary
vocab_path = config.data_dir / 'd3pm_vocab.json'
if vocab_path.exists():
    print(f'Loading pre-built vocab from {vocab_path}')
    tokenizer = MultiScaleTokenizer(vocab_file=str(vocab_path))
    print(f'✅ Vocabulary size: {tokenizer.vocab_size}')
else:
    print('No cached vocab found, building from data...')
    tokenizer = MultiScaleTokenizer()
    tokenizer.build_vocab(train_df['smiles'].tolist(), max_vocab_size=5000)
    # Save to data_dir for reuse
    tokenizer.save(str(vocab_path))

# Also save to run_dir for checkpoint completeness
config.model_dir.mkdir(parents=True, exist_ok=True)
tokenizer.save(str(config.model_dir / 'vocab.json'))

# Test encoding/decoding
test_smiles = train_df['smiles'].iloc[0]
print(f"\n🧪 Test encoding:")
print(f"Original: {test_smiles}")
encoded = tokenizer.encode(test_smiles)
print(f"Encoded: {encoded[:20]}... (length: {len(encoded)})")
decoded = tokenizer.decode(encoded)
print(f"Decoded: {decoded}")
print(f"Match: {test_smiles == decoded}")

In [ ]:
# CELL 11
class D3PMDataset(Dataset):
    """
    Advanced dataset with:
    - All extracted features
    - Contrastive pairs (positive/negative examples)
    - Property interpolation sequences
    - Stratified sampling
    
    FIX 15: Support passing normalization stats from train to val
    PERF FIX: Pre-cache scaffold tokens and numpy arrays for fast __getitem__
    """
    
    def __init__(self, df, tokenizer, extractor, max_len=128, 
                 contrastive_pairs=True, cfg_dropout=0.3, norm_stats=None):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.extractor = extractor
        self.max_len = max_len
        self.contrastive_pairs = contrastive_pairs
        self.cfg_dropout = cfg_dropout
        
        # FIX 15: Use provided norm_stats or compute from this dataset
        if norm_stats is not None:
            self.norm_stats = norm_stats
        else:
            # Compute normalization statistics
            self.norm_stats = {}
            prop_cols = ['qed', 'mw', 'logp', 'hbd', 'hba', 'psa', 'rotatable_bonds', 
                         'aromatic_rings', 'num_rings', 'num_heteroatoms', 'sp3_fraction']
            
            for col in prop_cols:
                if col in df.columns:
                    self.norm_stats[col] = {
                        'mean': df[col].mean(),
                        'std': df[col].std() + 1e-8
                    }

        # PERF FIX: Pre-cache scaffold tokens keyed by scaffold_id so __getitem__
        # doesn't call get_scaffold_smiles_safe + tokenizer.encode on every sample.
        print("🔧 Pre-caching scaffold tokens...")
        self._scaffold_token_cache = {}
        unique_scaffold_ids = df['scaffold_id'].unique()
        for sid in unique_scaffold_ids:
            smiles = get_scaffold_smiles_safe(extractor, int(sid), fallback_strategy='most_common')
            self._scaffold_token_cache[int(sid)] = self.tokenizer.encode(smiles, max_len=config.scaffold_max_len)
        print(f"✅ Cached {len(self._scaffold_token_cache)} scaffold token sequences")

        # PERF FIX: Pre-tokenize all molecule SMILES at init → O(1) numpy index per
        # __getitem__ instead of calling tokenizer.encode() ~160k times during training.
        print("🔧 Pre-tokenizing all SMILES (first run caches to disk)...")
        # CACHE FIX: Save token array to disk so subsequent runs are instant.
        _cache_key = f"tokens_n{len(self.df)}_len{max_len}_v{len(self.tokenizer.token_to_id)}"
        _cache_path = config.model_dir / f"{_cache_key}.npy"
        if _cache_path.exists():
            self._tokens = np.load(str(_cache_path))
            print(f"✅ Loaded token cache → shape {self._tokens.shape}")
        else:
            _t2id = self.tokenizer.token_to_id
            _bos = _t2id[self.tokenizer.bos_token]
            _eos = _t2id[self.tokenizer.eos_token]
            _pad = _t2id[self.tokenizer.pad_token]
            _unk = _t2id.get(self.tokenizer.unk_token, _pad)
            _max = max_len
            def _encode_one(smiles):
                import selfies as _sf
                try:
                    s = _sf.encoder(smiles)
                    ids = [_bos] + [_t2id.get(t, _unk) for t in _sf.split_selfies(s)] + [_eos]
                    if len(ids) > _max: ids = ids[:_max-1] + [_eos]
                    else: ids = ids + [_pad] * (_max - len(ids))
                    return ids
                except:
                    return [_pad] * _max
            smiles_list = self.df['smiles'].tolist()
            try:
                from joblib import Parallel, delayed
                n_jobs = min(4, os.cpu_count() or 1)
                print(f"   Parallel tokenizing with {n_jobs} workers...")
                results = Parallel(n_jobs=n_jobs, backend='loky')(
                    delayed(_encode_one)(s) for s in tqdm(smiles_list, desc='Tokenizing', leave=False)
                )
                self._tokens = np.array(results, dtype=np.int64)
            except Exception as _e:
                print(f"   joblib failed ({_e}), using sequential...")
                self._tokens = np.zeros((len(self.df), _max), dtype=np.int64)
                for _i, _s in enumerate(tqdm(smiles_list, desc='Tokenizing', leave=False)):
                    self._tokens[_i] = _encode_one(_s)
            np.save(str(_cache_path), self._tokens)
            print(f"✅ Pre-tokenized {len(self.df)} molecules → shape {self._tokens.shape} (cached to disk)")

        # Build contrastive pairs
        if contrastive_pairs:
            self._build_contrastive_pairs()
    
    def _build_contrastive_pairs(self):
        """Build positive/negative pairs for contrastive learning"""
        print("🔗 Building contrastive pairs...")
        
        # Group by QED bins — FIX 1B: Use actual number of QED bins (5), not hardcoded 4
        n_qed_bins = len(self.extractor.property_bins['qed']) - 1
        # PERF FIX: Store numpy arrays so np.random.choice is O(1) instead of
        # converting a Python list → numpy array on every __getitem__ call.
        self.qed_groups = {}
        for bin_id in range(n_qed_bins):
            self.qed_groups[bin_id] = np.array(
                self.df[self.df['qed_bin'] == bin_id].index.tolist(), dtype=np.int64
            )

        # PERF FIX: Pre-compute the list of "other bins" for each bin so we
        # don't rebuild it with a list comprehension on every __getitem__ call.
        self._other_bins = {
            bin_id: np.array([b for b in self.qed_groups.keys() if b != bin_id], dtype=np.int64)
            for bin_id in self.qed_groups.keys()
        }
        
        print(f"✅ Contrastive pairs ready ({n_qed_bins} QED bins)")
    
    def normalize_properties(self, props):
        """Normalize property values"""
        normalized = {}
        for key, value in props.items():
            if key in self.norm_stats:
                mean = self.norm_stats[key]['mean']
                std = self.norm_stats[key]['std']
                normalized[key] = (value - mean) / std
            else:
                normalized[key] = value
        return normalized
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # 1. Tokenize molecule
        tokens = self._tokens[idx]  # PERF FIX: pre-tokenized at init
        
        # 2. Property decomposition (QED components)
        properties = {
            'qed': row['qed'],
            'mw': row['mw'],
            'logp': row['logp'],
            'hbd': row['hbd'],
            'hba': row['hba'],
            'psa': row['psa'],
            'rotatable_bonds': row['rotatable_bonds'],
            'aromatic_rings': row['aromatic_rings'],
            'num_rings': row['num_rings'],
            'num_heteroatoms': row['num_heteroatoms'],
            'sp3_fraction': row['sp3_fraction'],
        }
        # FIX (R5): the property TENSOR is built from this dict's value order, and many
        # places assume QED == index 0. Assert the order matches config.property_list so a
        # reorder can never silently map properties to the wrong slots. Cached on the class
        # to keep the hot path cheap (checked once).
        if not getattr(D3PMDataset, '_prop_order_checked', False):
            assert list(properties.keys())[:len(config.property_list)] == list(config.property_list), (
                f"property dict order {list(properties.keys())} != config.property_list "
                f"{config.property_list}; QED-index-0 assumptions would break.")
            D3PMDataset._prop_order_checked = True
        
        # Normalize
        properties_norm = self.normalize_properties(properties)
        
        # 3. Discrete bins - FIX B5: Include all bins
        discrete_bins = {
            'qed_bin': row['qed_bin'],
            'mw_bin': row['mw_bin'],
            'logp_bin': row['logp_bin'],
            'hbd_bin': row.get('hbd_bin', 0),
            'hba_bin': row.get('hba_bin', 0),
            'psa_bin': row.get('psa_bin', 0),
        }
        
        # 4. Scaffold information — PERF FIX: use pre-cached tokens
        scaffold_id = row['scaffold_id']
        scaffold_tokens = self._scaffold_token_cache.get(
            int(scaffold_id),
            self._scaffold_token_cache.get(0, self.tokenizer.encode('', max_len=config.scaffold_max_len))
        )
        
        # 5. CFG dropout (classifier-free guidance)
        use_conditioning = torch.rand(1).item() > self.cfg_dropout
        
        # 6. Contrastive negative (different QED bin) — PERF FIX: use pre-built numpy arrays
        negative_idx = idx
        if self.contrastive_pairs and use_conditioning:
            current_bin = row['qed_bin']
            other_bins = self._other_bins.get(current_bin)
            if other_bins is not None and len(other_bins) > 0:
                neg_bin = int(np.random.choice(other_bins))
                neg_pool = self.qed_groups[neg_bin]
                if len(neg_pool) > 0:
                    negative_idx = int(np.random.choice(neg_pool))
        
        negative_row = self.df.iloc[negative_idx]
        negative_qed = negative_row['qed']
        
        return {
            'tokens': torch.LongTensor(tokens),
            'properties': torch.FloatTensor(list(properties_norm.values())),
            'properties_raw': torch.FloatTensor(list(properties.values())),
            'discrete_bins': torch.LongTensor(list(discrete_bins.values())),
            'scaffold_id': torch.LongTensor([scaffold_id]),
            'scaffold_tokens': torch.LongTensor(scaffold_tokens),  # FIX B6: Add tokenized scaffold
            'use_conditioning': torch.FloatTensor([1.0 if use_conditioning else 0.0]),
            'negative_qed': torch.FloatTensor([negative_qed]),
            'is_contrastive': torch.FloatTensor([1.0 if negative_idx != idx else 0.0]),
            'fragment_bits': torch.FloatTensor(
                row['fragment_bits'] if 'fragment_bits' in row.index 
                else np.zeros(MolecularFeatureExtractor.N_FRAGMENT_BITS, dtype=np.float32)
            ),
        }

print("✅ D3PMDataset created")


In [ ]:
# CELL 12
# Create datasets - FIX 15: Pass train normalization stats to val
train_dataset = D3PMDataset(
    train_df, 
    tokenizer, 
    extractor, 
    max_len=128,
    contrastive_pairs=False,  # FIX (R4): negative_qed/is_contrastive are never used by
    #            the in-batch InfoNCE loss; building them per __getitem__ was pure wasted
    #            work in the hot path. InfoNCE still runs (it uses h_pooled + prop encoding).
    cfg_dropout=0.3,
    norm_stats=None  # Compute from train
)

# FIX 15: Use train normalization stats for val
val_dataset = D3PMDataset(
    val_df, 
    tokenizer, 
    extractor, 
    max_len=128,
    contrastive_pairs=False,
    cfg_dropout=0.0,
    norm_stats=train_dataset.norm_stats  # FIX 15: Use train stats
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=64, 
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=64, 
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"✅ Training batches: {len(train_loader)}")
print(f"✅ Validation batches: {len(val_loader)}")

# Test batch
test_batch = next(iter(train_loader))
print(f"\n🧪 Test batch shapes:")
for key, value in test_batch.items():
    print(f"  {key}: {value.shape}")

# Save normalization stats
with open(config.model_dir / 'norm_stats.json', 'w') as f:
    json.dump(train_dataset.norm_stats, f)

print("\n💾 Normalization stats saved")

In [ ]:
# CELL 13
# ============================================================================
# SHARED GENERATION / RL / EVAL HELPERS  (R1 fix: moved up from cell ~67 so a
# top-to-bottom run defines them BEFORE evaluate_generator and the comparison cells)
# ============================================================================

def forward_diffuse_absorbing(x0, t_batch, model, tokenizer):
    """
    D3PM absorbing-state forward diffusion.
    Each token becomes [MASK] with probability (1 - alpha_bar[t]).
    BOS, PAD, EOS positions are preserved.
    """
    device = x0.device
    mask_prob = 1.0 - model.alpha_bars[t_batch]          # (B,)
    rand = torch.rand_like(x0.float())                   # (B, L)
    should_mask = rand < mask_prob.unsqueeze(1)           # (B, L)

    # Preserve special tokens
    bos_pos = (x0 == tokenizer.token_to_id['[BOS]'])
    pad_pos = (x0 == tokenizer.token_to_id['[PAD]'])
    eos_pos = (x0 == tokenizer.token_to_id['[EOS]'])
    should_mask = should_mask & ~bos_pos & ~pad_pos & ~eos_pos

    xt = x0.clone()
    xt[should_mask] = tokenizer.token_to_id['[MASK]']
    return xt

# ── Reward function ──
def compute_qed_rewards(generated_tokens, target_qed, tokenizer):
    """Compute reward = -|actual_QED - target_QED|. Invalid molecules get -2.0."""
    rewards = []
    actual_qeds = []
    valid_smiles = []
    for tok in generated_tokens:
        smiles = tokenizer.decode(tok.cpu().tolist())
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            try:
                qed_val = QED.qed(mol)
                rewards.append(-abs(qed_val - target_qed))
                actual_qeds.append(qed_val)
                valid_smiles.append(Chem.MolToSmiles(mol))
            except Exception:
                rewards.append(-2.0)
                actual_qeds.append(None)
                valid_smiles.append(None)
        else:
            rewards.append(-2.0)
            actual_qeds.append(None)
            valid_smiles.append(None)
    return (
        torch.tensor(rewards, device=generated_tokens.device, dtype=torch.float32),
        actual_qeds,
        valid_smiles,
    )

# ── Build conditioning tensors helper ──
def build_conditioning_for_qed(target_qed, dataset, extractor, device):
    """Build all conditioning tensors for a given target QED."""
    similar = dataset.df[
        (dataset.df['qed'] >= target_qed - 0.15) &
        (dataset.df['qed'] <= target_qed + 0.15)
    ]
    if len(similar) > 0:
        ref = similar.sample(1).iloc[0]
        target_props_raw = {
            'qed': target_qed, 'mw': ref['mw'], 'logp': ref['logp'],
            'hbd': ref['hbd'], 'hba': ref['hba'], 'psa': ref['psa'],
            'rotatable_bonds': ref['rotatable_bonds'],
            'aromatic_rings': ref['aromatic_rings'],
            'num_rings': ref['num_rings'],
            'num_heteroatoms': ref['num_heteroatoms'],
            'sp3_fraction': ref['sp3_fraction'],
        }
        scaffold_id = int(ref['scaffold_id'])
    else:
        target_props_raw = {
            'qed': target_qed, 'mw': 350.0, 'logp': 2.5, 'hbd': 2.0,
            'hba': 4.0, 'psa': 70.0, 'rotatable_bonds': 5.0,
            'aromatic_rings': 2.0, 'num_rings': 3.0,
            'num_heteroatoms': 4.0, 'sp3_fraction': 0.3,
        }
        scaffold_id = int(np.random.choice(list(dataset.df['scaffold_id'].unique())))

    target_props_norm = dataset.normalize_properties(target_props_raw)
    target_tensor = torch.FloatTensor(list(target_props_norm.values())).to(device)

    qed_bin, _ = extractor.discretize_property(target_qed, 'qed')
    mw_bin,  _ = extractor.discretize_property(target_props_raw['mw'],  'mw')
    logp_bin,_ = extractor.discretize_property(target_props_raw['logp'],'logp')
    hbd_bin, _ = extractor.discretize_property(target_props_raw['hbd'], 'hbd')
    hba_bin, _ = extractor.discretize_property(target_props_raw['hba'], 'hba')
    psa_bin, _ = extractor.discretize_property(target_props_raw['psa'], 'psa')
    discrete_bins = torch.LongTensor([qed_bin, mw_bin, logp_bin, hbd_bin, hba_bin, psa_bin]).to(device)

    return target_tensor, scaffold_id, discrete_bins


print("\u2705 Shared generation/RL/eval helpers defined (build_conditioning_for_qed, "
      "compute_qed_rewards, forward_diffuse_absorbing)")


## 7 · CondDiT model

DiT + adaLN-Zero. Generic in `n_props`; instantiated with **n_props=1 (QED-only)**.

In [ ]:
# CELL 14
# CondDiT — conditional diffusion transformer (DiT backbone + adaLN-Zero).
# Generic in n_props; we instantiate with n_props=1 (QED-only conditioning).
import math, torch, torch.nn as nn, torch.nn.functional as F

def modulate(x, shift, scale):
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

class TimestepEmbed(nn.Module):
    def __init__(self, d):
        super().__init__(); self.d = d
        self.mlp = nn.Sequential(nn.Linear(d, d), nn.SiLU(), nn.Linear(d, d))
    def forward(self, t):
        half = self.d // 2
        f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        a = t.float().unsqueeze(1) * f.unsqueeze(0)
        e = torch.cat([torch.cos(a), torch.sin(a)], -1)
        if self.d % 2: e = F.pad(e, (0, 1))
        return self.mlp(e)

class DiTBlock(nn.Module):
    def __init__(self, d, nh, dr=0.1):
        super().__init__()
        self.n1 = nn.LayerNorm(d, elementwise_affine=False, eps=1e-6)
        self.attn = nn.MultiheadAttention(d, nh, dropout=dr, batch_first=True)
        self.n2 = nn.LayerNorm(d, elementwise_affine=False, eps=1e-6)
        self.mlp = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
        self.ada = nn.Sequential(nn.SiLU(), nn.Linear(d, 6*d))
        nn.init.zeros_(self.ada[-1].weight); nn.init.zeros_(self.ada[-1].bias)
    def forward(self, x, c, pad_mask):
        sm, scm, gm, sp, scp, gp = self.ada(c).chunk(6, -1)
        h = modulate(self.n1(x), sm, scm)
        a, _ = self.attn(h, h, h, key_padding_mask=pad_mask, need_weights=False)
        x = x + gm.unsqueeze(1) * a
        h = modulate(self.n2(x), sp, scp)
        x = x + gp.unsqueeze(1) * self.mlp(h)
        return x

class CondDiT(nn.Module):
    def __init__(self, V, max_len, n_props, d=512, nh=8, nl=6, T=100, dr=0.1,
                 pad_id=0, bos_id=1, eos_id=2, mask_id=3):
        super().__init__()
        self.tok = nn.Embedding(V, d); self.pos = nn.Embedding(max_len, d); self.te = TimestepEmbed(d)
        self.pm = nn.Sequential(nn.Linear(n_props, 4*d), nn.SiLU(), nn.Linear(4*d, d))  # wider bottleneck for weak scalar signal
        self.null = nn.Parameter(torch.zeros(n_props))
        self.blocks = nn.ModuleList([DiTBlock(d, nh, dr) for _ in range(nl)])
        self.nf = nn.LayerNorm(d, elementwise_affine=False, eps=1e-6)
        self.af = nn.Sequential(nn.SiLU(), nn.Linear(d, 2*d))
        nn.init.zeros_(self.af[-1].weight); nn.init.zeros_(self.af[-1].bias)
        self.head = nn.Linear(d, V)
        self.T = T; self.max_len = max_len; self.pad_id = pad_id; self.bos_id = bos_id
        self.eos_id = eos_id; self.mask_id = mask_id; self.vocab_size = V; self.n_props = n_props
        s = torch.arange(T+1)/T; ab = torch.cos((s+0.008)/1.008 * math.pi/2)**2
        self.register_buffer('alpha_bars', (ab/ab[0])[1:])
    def q_sample(self, x0, t, mask_id=None):
        mid = self.mask_id if mask_id is None else mask_id; B, L = x0.shape
        valid = (x0 != self.pad_id) & (x0 != self.bos_id)
        sm = (torch.rand(B, L, device=x0.device) < (1 - self.alpha_bars[t].view(-1, 1))) & valid
        xt = x0.clone(); xt[sm] = mid
        return xt, (xt == mid)
    def forward(self, xt, t, props, use_cond, pad_mask):
        B, L = xt.shape
        pos = torch.arange(L, device=xt.device).unsqueeze(0).expand(B, -1)
        h = self.tok(xt) + self.pos(pos)
        # use_cond: (B,) all-or-nothing keep (back-compat) OR (B, n_props) per-property keep-mask
        if use_cond.dim() == 1:
            cmask = use_cond.view(B, 1).float().expand(B, self.n_props)
        else:
            cmask = use_cond.float()
        p = cmask * props + (1 - cmask) * self.null.unsqueeze(0)
        c = self.te(t) + self.pm(p)
        for blk in self.blocks: h = blk(h, c, pad_mask)
        sh, sc = self.af(c).chunk(2, -1); h = modulate(self.nf(h), sh, sc)
        return self.head(h)

print("✅ CondDiT defined (instantiate with n_props=1 for QED-only)")

## 8 · Setup — Drive dirs, QED conditioning, W&B

Results dirs on Drive, `qed_to_cond` (z-score normalize a target QED), and the W&B initializer (graceful — trains without it if no key).

In [ ]:
# CELL 15
# Drive results dirs + QED-only conditioning helper + Weights & Biases (optional).
import os, json, math, time
from pathlib import Path
import numpy as np

RESULTS_DIR = config.model_dir / 'dit_qedonly'
CKPT_DIR    = RESULTS_DIR / 'checkpoints'
GEN_DIR     = RESULTS_DIR / 'generations'
for d in (RESULTS_DIR, CKPT_DIR, GEN_DIR): d.mkdir(parents=True, exist_ok=True)
print("📁 everything saves under:", RESULTS_DIR)

PAD = tokenizer.token_to_id['[PAD]']; BOS = tokenizer.token_to_id['[BOS]']
EOS = tokenizer.token_to_id['[EOS]']; MASKi = tokenizer.token_to_id['[MASK]']

# QED-only conditioning: normalize a raw target QED exactly how the dataset does (z-score).
QMEAN = float(train_dataset.norm_stats['qed']['mean'])
QSTD  = float(train_dataset.norm_stats['qed']['std'])
def qed_to_cond(target_qed):
    return torch.tensor([(target_qed - QMEAN) / QSTD], dtype=torch.float32, device=device)
print(f"QED norm: mean={QMEAN:.3f} std={QSTD:.3f}  →  conditioning on QED ONLY (n_props=1)")

# ---- Weights & Biases (graceful: trains fine without it) ----
def init_wandb(run_name, cfg_dict):
    try:
        import wandb
    except ImportError:
        print("ℹ️ wandb not installed (pip install wandb) — training runs without it"); return None
    # Key from env, then python-dotenv, then a .env file searched in cwd + Google Drive.
    # (Same mechanism as the old D3PM notebook: your .env on Drive holds WANDB_API_KEY.)
    key = os.environ.get("WANDB_API_KEY")
    if not key:
        try:
            from dotenv import load_dotenv, find_dotenv
            load_dotenv(find_dotenv(usecwd=True)); key = os.environ.get("WANDB_API_KEY")
        except Exception:
            pass
    if not key:
        for base in [Path.cwd(), *Path.cwd().parents, Path("/content/drive/MyDrive"),
                     Path("/content/drive/MyDrive/Datasets/Drug Discovery")]:
            try:
                envf = base / ".env"
                if envf.exists():
                    for line in envf.read_text().splitlines():
                        line = line.strip()
                        if line.startswith("WANDB_API_KEY") and "=" in line:
                            key = line.split("=", 1)[1].strip().strip("'").strip('"'); break
                if key:
                    break
            except Exception:
                continue
    if not key:
        print("⚠️ no WANDB_API_KEY (env or .env on Drive) — training runs WITHOUT wandb"); return None
    os.environ["WANDB_API_KEY"] = key
    try:
        wandb.login(key=key)
        return wandb.init(project="dit-conditional-molgen", entity=os.environ.get("WANDB_ENTITY"),
                          name=run_name, config=cfg_dict, reinit=True)
    except Exception as e:
        print("wandb init failed (continuing without):", e); return None

print("✅ setup ready")

## 9 · Training — QED-only, resume, save-every-epoch, W&B

Saves `latest.pt` + `best.pt` + `history.json` **every epoch**, plus a permanent `epoch_NNN.pt` **every `MILESTONE_EVERY` (=10) epochs**. `RESUME_FROM` continues from any checkpoint (latest, or a specific milestone).

In [ ]:
# CELL 16
# QED-only CondDiT training. Resumes from any checkpoint, saves EVERY epoch + best + latest to Drive, logs to W&B.
EPOCHS           = 200
LR               = 3e-4
CFG_DROPOUT      = 0.3
RESUME_FROM      = 'latest'     # 'latest' | 'fresh' | a path e.g. str(CKPT_DIR/'epoch_040.pt')
MILESTONE_EVERY = 10           # permanent epoch_NNN.pt every N epochs (latest + best saved EVERY epoch)
amp = torch.cuda.is_available()

dit = CondDiT(tokenizer.vocab_size, 128, n_props=1, d=512, nh=8, nl=6, T=100,
              pad_id=PAD, bos_id=BOS, eos_id=EOS, mask_id=MASKi).to(device)
opt = torch.optim.AdamW(dit.parameters(), lr=LR, weight_decay=0.01)
start_epoch, best, history = 0, float('inf'), []

def _load_resume(p):
    global start_epoch, best, history
    ck = torch.load(p, map_location=device)
    dit.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    start_epoch = ck['epoch']; best = ck.get('best', float('inf')); history = ck.get('history', [])
    print(f"▶ RESUMED from {Path(p).name}: epoch {start_epoch}, best val_realCE {best:.3f}")

if RESUME_FROM == 'latest' and (CKPT_DIR/'latest.pt').exists():
    _load_resume(CKPT_DIR/'latest.pt')
elif RESUME_FROM not in ('latest', 'fresh') and Path(RESUME_FROM).exists():
    _load_resume(RESUME_FROM)
else:
    print(f"starting FRESH: {sum(p.numel() for p in dit.parameters()):,} params, QED-only conditioning")

wb = init_wandb("qedonly_d512_l6", {'epochs': EPOCHS, 'lr': LR, 'cfg_dropout': CFG_DROPOUT, 'cond': 'qed_only', 'd': 512, 'layers': 6})

@torch.no_grad()
def _val(m, nb=16):
    m.eval(); ce = cn = acc = ec = en = 0
    for i, batch in enumerate(val_loader):
        if i >= nb: break
        x0 = batch['tokens'].to(device); q = batch['properties'][:, 0:1].to(device).nan_to_num(0.0)
        t = torch.randint(0, m.T, (x0.size(0),), device=device); xt, im = m.q_sample(x0, t)
        logits = m(xt, t, q, torch.ones(x0.size(0), device=device), pad_mask=(xt == PAD))
        real = im & (x0 != PAD) & (x0 != BOS) & (x0 != EOS)
        if real.any():
            ce += F.cross_entropy(logits[real].float(), x0[real], reduction='sum').item()
            acc += (logits[real].argmax(-1) == x0[real]).sum().item(); cn += real.sum().item()
        em = im & (x0 == EOS)
        if em.any(): ec += (logits[em].argmax(-1) == x0[em]).sum().item(); en += em.sum().item()
    m.train(); return ce/max(cn, 1), acc/max(cn, 1), ec/max(en, 1)

def _payload(ep):
    return {'model': dit.state_dict(), 'opt': opt.state_dict(), 'epoch': ep, 'best': best,
            'history': history, 'n_props': 1, 'qed_mean': QMEAN, 'qed_std': QSTD}

print(f"{'ep':>3} | {'val_realCE':>10} | {'ppl':>5} | {'acc':>5} | {'eos':>5} | {'sec':>4}")
for ep in range(start_epoch + 1, EPOCHS + 1):
    dit.train(); t0 = time.time()
    for batch in train_loader:
        x0 = batch['tokens'].to(device); B = x0.size(0)
        q = batch['properties'][:, 0:1].to(device).nan_to_num(0.0)
        t = torch.randint(0, dit.T, (B,), device=device); xt, im = dit.q_sample(x0, t)
        uc = (torch.rand(B, device=device) > CFG_DROPOUT).float()
        opt.zero_grad()
        with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=amp):
            logits = dit(xt, t, q, uc, pad_mask=(xt == PAD))
            if not im.any(): continue
            loss = F.cross_entropy(logits[im].float(), x0[im])
        loss.backward(); torch.nn.utils.clip_grad_norm_(dit.parameters(), 1.0); opt.step()
    vce, vacc, veos = _val(dit); sec = time.time() - t0
    rec = {'epoch': ep, 'val_realCE': vce, 'ppl': math.exp(vce), 'acc': vacc, 'eos_acc': veos, 'sec': sec}
    history.append(rec)
    print(f"{ep:>3} | {vce:>10.3f} | {math.exp(vce):>5.1f} | {vacc:>5.3f} | {veos:>5.3f} | {sec:>4.0f}")
    torch.save(_payload(ep), CKPT_DIR/'latest.pt')
    if ep % MILESTONE_EVERY == 0: torch.save(_payload(ep), CKPT_DIR/f'epoch_{ep:03d}.pt')
    if vce < best:
        best = vce; torch.save(_payload(ep), CKPT_DIR/'best.pt'); print(f"     ✅ best (val_realCE {vce:.3f}) saved")
    json.dump(history, open(RESULTS_DIR/'history.json', 'w'), indent=1)
    if wb: wb.log(rec)
if wb: wb.finish()
print("✅ training complete. checkpoints + history.json under", RESULTS_DIR)

## 10 · CFG sampler

MaskGIT reverse diffusion + CFG + min-length guard.

In [ ]:
# CELL 17
# CFG MaskGIT sampler with min-length guard (forbids empty molecules). Generic in props.
@torch.no_grad()
def sample(dit, props, n=64, steps=50, cfg=2.0, temperature=1.0, noise=1.0, min_len=12, cmask=None):
    dit.eval(); L = dit.max_len
    xt = torch.full((n, L), dit.mask_id, device=device); xt[:, 0] = dit.bos_id
    P = props.to(device); P = P.unsqueeze(0).expand(n, -1) if P.dim() == 1 else P
    if cmask is None:
        on = torch.ones(n, device=device)
    else:
        _cm = cmask.to(device).float(); on = _cm.unsqueeze(0).expand(n, -1) if _cm.dim()==1 else _cm
    off = torch.zeros(n, device=device)
    n0 = L - 1
    for i in range(steps):
        pad_mask = (xt == dit.pad_id)
        t = torch.full((n,), int(round((1 - i/steps) * (dit.T - 1))), device=device)
        cond = dit(xt, t, P, on, pad_mask)
        if cfg == 1.0:
            logits = cond
        else:
            unc = dit(xt, t, P, off, pad_mask); logits = unc + cfg * (cond - unc)
        logits[:, :, dit.mask_id] = -1e9
        logits[:, 1:, dit.bos_id] = -1e9
        if min_len > 0: logits[:, :min_len, dit.eos_id] = -1e9
        probs = F.softmax(logits / temperature, -1)
        pred = torch.multinomial(probs.reshape(-1, probs.size(-1)), 1).reshape(n, L)
        conf = probs.gather(-1, pred.unsqueeze(-1)).squeeze(-1)
        mask = (xt == dit.mask_id); conf = conf.masked_fill(~mask, -1e9)
        if noise > 0:
            u = torch.rand_like(conf).clamp(1e-9, 1-1e-9); g = -torch.log(-torch.log(u))
            conf = conf + noise * (1 - i/steps) * g
        n_keep = int(round(n0 * math.cos(math.pi/2 * (i+1)/steps)))
        to_reveal = int(mask[0].sum().item()) - n_keep
        if to_reveal > 0:
            idx = conf.topk(to_reveal, dim=1).indices
            r = torch.arange(n, device=device).unsqueeze(1).expand(-1, to_reveal)
            xt[r, idx] = pred[r, idx]
        if not (xt == dit.mask_id).any(): break
    if (xt == dit.mask_id).any():
        t = torch.zeros(n, dtype=torch.long, device=device)
        lg = dit(xt, t, P, on, (xt == dit.pad_id)); lg[:, :, dit.mask_id] = -1e9
        m = (xt == dit.mask_id); xt = torch.where(m, lg.argmax(-1), xt)
    return xt
print("✅ sampler ready (min-length guard)")

## 11 · Load any checkpoint

`load_checkpoint('best' | 'latest' | epoch_int | path)` → rebuilds `dit`.

In [ ]:
# CELL 18
# Load ANY saved checkpoint into `dit` — for eval, or before re-running the train cell to continue.
def load_checkpoint(which='best'):
    """which: 'best' | 'latest' | epoch int (e.g. 40) | full path string."""
    if isinstance(which, int): p = CKPT_DIR / f'epoch_{which:03d}.pt'
    elif which in ('best', 'latest'): p = CKPT_DIR / f'{which}.pt'
    else: p = Path(which)
    assert p.exists(), f"checkpoint not found: {p}"
    global dit
    dit = CondDiT(tokenizer.vocab_size, 128, n_props=1, pad_id=PAD, bos_id=BOS, eos_id=EOS, mask_id=MASKi).to(device)
    ck = torch.load(p, map_location=device); dit.load_state_dict(ck['model']); dit.eval()
    print(f"✅ loaded {p.name}: epoch {ck.get('epoch','?')}, best val_realCE {ck.get('best','?')}")
    return dit

# After training, load the best checkpoint for evaluation:
load_checkpoint('best')

## 12 · Steering evaluation

QED-only steering table; saved to `steering_results.json`.

In [ ]:
# CELL 19
# QED-steering evaluation (QED-only conditioning). Saves the table to Drive.
from rdkit import Chem
from rdkit.Chem import QED
import numpy as np, json

@torch.no_grad()
def evaluate_steering(dit, targets=(0.3, 0.5, 0.7, 0.9), n=256, cfg=0.6, steps=50, save=True):
    rows = []
    print(f"{'target':>7} | {'valid':>7} | {'uniq':>5} | {'meanQED':>8} | {'meanHA':>7} | {'>0.7':>6}")
    print("-"*58)
    for tq in targets:
        cond = qed_to_cond(tq); qeds, has, smis = [], [], set()
        for _ in range(max(1, n // 128)):
            for tk in sample(dit, cond, n=128, cfg=cfg, steps=steps):
                mol = Chem.MolFromSmiles(tokenizer.decode(tk.cpu().tolist()))
                if mol is None: continue
                try: qeds.append(QED.qed(mol)); has.append(mol.GetNumHeavyAtoms()); smis.add(Chem.MolToSmiles(mol))
                except Exception: pass
        q = np.array(qeds) if qeds else np.array([0.0])
        row = {'target': tq, 'valid': len(qeds), 'uniq': len(smis)/max(len(qeds), 1),
               'meanQED': float(q.mean()), 'meanHA': float(np.mean(has) if has else 0), 'frac>0.7': float((q > 0.7).mean())}
        rows.append(row)
        print(f"{tq:>7.2f} | {len(qeds):>7} | {row['uniq']:>5.2f} | {row['meanQED']:>8.3f} | {row['meanHA']:>7.1f} | {row['frac>0.7']:>6.1%}")
    if save:
        json.dump(rows, open(RESULTS_DIR/'steering_results.json', 'w'), indent=1)
        print("💾 saved →", RESULTS_DIR/'steering_results.json')
    print("\nWIN: meanQED rises with target (centered well above 0.7 at target 0.9).")
    return rows

evaluate_steering(dit, cfg=0.6)

## 13 · Best-of-N generation

Generate at a target QED, rank by RDKit QED, **save molecules to CSV** on Drive.

In [ ]:
# CELL 20
# Generate at a target QED, rank by RDKit QED, SAVE the molecules to Drive (CSV).
import csv
@torch.no_grad()
def best_of_n(dit, target_qed=0.9, n_gen=512, keep=50, cfg=0.6, min_ha=8, save=True):
    cond = qed_to_cond(target_qed); rows = []
    for _ in range(max(1, n_gen // 128)):
        for tk in sample(dit, cond, n=128, cfg=cfg):
            mol = Chem.MolFromSmiles(tokenizer.decode(tk.cpu().tolist()))
            if mol is None or mol.GetNumHeavyAtoms() < min_ha: continue
            try: rows.append((QED.qed(mol), mol.GetNumHeavyAtoms(), Chem.MolToSmiles(mol)))
            except Exception: pass
    rows = sorted(set(rows), key=lambda r: -r[0])[:keep]
    if not rows: print("no valid molecules ≥ min_ha"); return rows
    print(f"target {target_qed}: kept {len(rows)} | best QED {rows[0][0]:.3f} | median {np.median([r[0] for r in rows]):.3f}\n")
    for q, ha, smi in rows[:12]: print(f"  QED={q:.3f}  HA={ha:>2}  {smi}")
    if save:
        p = GEN_DIR / f'best_of_n_qed{target_qed}.csv'
        with open(p, 'w', newline='') as f:
            w = csv.writer(f); w.writerow(['qed', 'heavy_atoms', 'smiles'])
            for q, ha, smi in rows: w.writerow([f"{q:.4f}", ha, smi])
        print("\n💾 saved →", p)
    return rows

best_of_n(dit, target_qed=0.9, n_gen=512, keep=50, cfg=0.6)

## 14 · Quality-control pass

Validity, uniqueness, **novelty** (vs training set), **SA score** (synthesizability), internal diversity, distribution match. Saved to `qc_report.json`.

In [ ]:
# CELL 21
# Quality-control pass: validity, uniqueness, NOVELTY, synthesizability (SA), diversity, distribution match.
from rdkit.Chem import AllChem, DataStructs, RDConfig
import sys, os
try:
    sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score')); import sascorer; _HAS_SA = True
except Exception as _e:
    print("SA scorer unavailable:", _e); _HAS_SA = False

_TRAIN_SMILES = set(train_dataset.df['canonical_smiles'])
try:
    _DATA_HA = float(np.mean([Chem.MolFromSmiles(s).GetNumHeavyAtoms()
                              for s in train_dataset.df['canonical_smiles'].sample(min(300, len(train_dataset.df)))
                              if Chem.MolFromSmiles(s) is not None]))
except Exception:
    _DATA_HA = float('nan')

@torch.no_grad()
def qc_report(dit, target_qed=0.9, n=512, cfg=0.6, save=True):
    cond = qed_to_cond(target_qed); mols = []; n_attempt = 0
    for _ in range(max(1, n // 128)):
        n_attempt += 128
        for tk in sample(dit, cond, n=128, cfg=cfg):
            mol = Chem.MolFromSmiles(tokenizer.decode(tk.cpu().tolist()))
            if mol is not None: mols.append(mol)
    if not mols: print("no valid molecules"); return {}
    smis = [Chem.MolToSmiles(m) for m in mols]; uniq = set(smis)
    qeds = np.array([QED.qed(m) for m in mols]); has = np.array([m.GetNumHeavyAtoms() for m in mols])
    sas = np.array([sascorer.calculateScore(m) for m in mols]) if _HAS_SA else np.array([float('nan')])
    novel = sum(1 for s in uniq if s not in _TRAIN_SMILES) / max(len(uniq), 1)
    fps = [AllChem.GetMorganFingerprintAsBitVect(m, 2, 1024) for m in mols[:200]]
    sims = [DataStructs.TanimotoSimilarity(fps[i], fps[j]) for i in range(len(fps)) for j in range(i+1, len(fps))]
    intdiv = 1 - (float(np.mean(sims)) if sims else 0.0)
    rep = {'target_qed': target_qed, 'n_attempted': n_attempt, 'n_valid': len(mols),
           'validity': len(mols)/n_attempt, 'uniqueness': len(uniq)/len(mols), 'novelty': novel,
           'mean_qed': float(qeds.mean()), 'frac_qed>0.7': float((qeds > 0.7).mean()),
           'mean_SA': float(np.nanmean(sas)), 'frac_SA<4.5': float(np.mean(sas < 4.5)) if _HAS_SA else None,
           'mean_HA': float(has.mean()), 'internal_diversity': intdiv,
           'data_mean_qed': float(train_dataset.df['qed'].mean()), 'data_mean_HA': _DATA_HA}
    print("="*52); print(f"  QC REPORT  (target QED {target_qed}, cfg {cfg})"); print("="*52)
    for k, v in rep.items():
        print(f"  {k:18s}: {v:.3f}" if isinstance(v, float) else f"  {k:18s}: {v}")
    print("\n  read: validity↑  novelty↑ (not memorized)  SA<4.5 = easy to make  internal_diversity↑ (not mode-collapsed)")
    if save:
        json.dump(rep, open(RESULTS_DIR/'qc_report.json', 'w'), indent=1); print("\n💾 saved →", RESULTS_DIR/'qc_report.json')
    return rep

qc_report(dit, target_qed=0.9, cfg=0.6)

## 15 · Phase 2 — Multi-property (9-prop) base + per-property dropout
<sub>9 props: qed, mw, hbd, hba, psa, num_rings, sa_score, hERG, pchembl_mean. Per-property dropout p=0.3 + ~10% all-null. Run after CELL 13. Gate A = fast-loop switch inside CELL 24; Gate B = CELL 25.</sub>

In [ ]:
# ============================================================================
# CELL 22 · PHASE 2 (1/4) — data-plumbing merge  (run AFTER CELL 13)
# df_enriched was rebuilt from the feature extractor and DROPPED the raw
# ADMET/bioactivity columns. Merge sa_score, hERG, pchembl_mean back onto
# train_df / val_df, keyed on SMILES.  Idempotent: safe to re-run.
# ============================================================================
import pandas as pd, numpy as np

STAGE2_PROPS = ['qed','mw','hbd','hba','psa','num_rings','sa_score','hERG','pchembl_mean']
MERGE_COLS   = ['sa_score','hERG','pchembl_mean']   # the 3 missing from df_enriched

_raw = pd.read_csv(config.data_dir / 'molecules_200k_selfies_clean.csv',
                   usecols=['canonical_smiles'] + MERGE_COLS)
_raw = _raw.drop_duplicates(subset='canonical_smiles', keep='first') \
           .rename(columns={'canonical_smiles': 'smiles'})

def _merge_props(df):
    df = df.copy()
    drop = [c for c in MERGE_COLS if c in df.columns]    # idempotent re-merge
    if drop: df = df.drop(columns=drop)
    return df.merge(_raw, on='smiles', how='left')

assert 'smiles' in train_df.columns, "train_df has no 'smiles' column to merge on"
train_df = _merge_props(train_df)
val_df   = _merge_props(val_df)

# all 9 conditioning columns must now exist
for _c in STAGE2_PROPS:
    assert _c in train_df.columns, f"MISSING conditioning column after merge: {_c}"
    assert _c in val_df.columns,   f"MISSING conditioning column after merge: {_c}"

# LOUD failure if the SMILES join key didn't match (canonicalisation mismatch etc.)
_herg_cov = train_df['hERG'].notna().mean()
assert _herg_cov > 0.9, (
    f"merge key mismatch: hERG only {_herg_cov*100:.1f}% non-null after join "
    f"(expected ~100%). train_df['smiles'] likely differs from raw canonical_smiles.")

print("merged. coverage on train_df:")
for _c in STAGE2_PROPS:
    print(f"   {_c:14s} {train_df[_c].notna().mean()*100:5.1f}% non-null")
print(f"\ntrain_df rows={len(train_df)}  val_df rows={len(val_df)}")


In [ ]:
# ============================================================================
# CELL 23 · PHASE 2 (2/4) — Stage2Dataset (9-prop + per-property mask)
# Self-contained; does NOT extend the heavy D3PMDataset.
# Reuses the already-built token cache (positionally aligned with train_df/val_df).
#   props = z-scored 9-vector ; pmask[k]=1 if value present else 0 (-> null at use).
#   pchembl_mean normalised over its non-null subset only (21% coverage).
# ============================================================================
class Stage2Dataset(Dataset):
    PROPS = STAGE2_PROPS   # canonical property order for the WHOLE project

    def __init__(self, df, tokens, norm_stats=None):
        assert len(df) == len(tokens), f"df/tokens length mismatch {len(df)} vs {len(tokens)}"
        self.df = df.reset_index(drop=True)
        self.tokens = tokens
        if norm_stats is None:
            self.norm_stats = {}
            for c in self.PROPS:
                vals = self.df[c].astype('float32').dropna()
                self.norm_stats[c] = {'mean': float(vals.mean()),
                                      'std':  float(vals.std() + 1e-8)}
        else:
            self.norm_stats = norm_stats
        # vectorised precompute -> O(1) __getitem__
        P = np.zeros((len(self.df), len(self.PROPS)), dtype=np.float32)
        M = np.zeros((len(self.df), len(self.PROPS)), dtype=np.float32)
        for j, c in enumerate(self.PROPS):
            raw = self.df[c].to_numpy(dtype=np.float32)        # NaN where missing
            present = ~np.isnan(raw)
            mu, sd = self.norm_stats[c]['mean'], self.norm_stats[c]['std']
            P[:, j] = (np.nan_to_num(raw, nan=mu) - mu) / sd   # missing -> 0 (masked anyway)
            M[:, j] = present.astype(np.float32)
        self._P, self._M = P, M

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            'tokens': torch.from_numpy(self.tokens[idx].astype(np.int64)),
            'props':  torch.from_numpy(self._P[idx]),
            'pmask':  torch.from_numpy(self._M[idx]),
        }

print("Stage2Dataset defined (9 props:", STAGE2_PROPS, ")")


In [ ]:
# ============================================================================
# CELL 24 · PHASE 2 (3/4) — per-property-dropout training (Gate A fast-loop switch)
#   GATE A: leave FAST_LOOP=True -> ~3-min fast-loop. Confirm val_realCE trends
#   toward ~1.3 and eos rises. THEN set FAST_LOOP=False and re-run for full train.
# ============================================================================
import time, math, json

EPOCHS_S2    = 200
LR_S2        = 3e-4
PROP_DROP    = 0.3      # per-property independent dropout
ALLNULL_PROB = 0.10     # fraction of steps with the whole row nulled (CFG null path)
FAST_LOOP    = False     # <<< GATE A switch
FAST_STEPS   = 300

S2_DIR  = config.model_dir / 'dit_9prop'
S2_CKPT = S2_DIR / 'checkpoints'; S2_CKPT.mkdir(parents=True, exist_ok=True)
amp = torch.cuda.is_available()

# --- datasets / loaders (reuse positionally-aligned token caches) ---
train_ds2 = Stage2Dataset(train_df, train_dataset._tokens)
val_ds2   = Stage2Dataset(val_df,   val_dataset._tokens, norm_stats=train_ds2.norm_stats)
json.dump(train_ds2.norm_stats, open(config.model_dir/'stage2_norm_stats.json','w'), indent=1)
train_loader2 = DataLoader(train_ds2, batch_size=64, shuffle=True,  num_workers=0, pin_memory=True)
val_loader2   = DataLoader(val_ds2,   batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

dit9 = CondDiT(tokenizer.vocab_size, 128, n_props=9, d=512, nh=8, nl=6, T=100,
               pad_id=PAD, bos_id=BOS, eos_id=EOS, mask_id=MASKi).to(device)
opt9 = torch.optim.AdamW(dit9.parameters(), lr=LR_S2, weight_decay=0.01)
print(f"9-prop CondDiT: {sum(p.numel() for p in dit9.parameters()):,} params")

@torch.no_grad()
def _val9(m, nb=16):
    m.eval(); ce=cn=acc=ec=en=0
    for i,b in enumerate(val_loader2):
        if i>=nb: break
        x0=b['tokens'].to(device); props=b['props'].to(device); pmask=b['pmask'].to(device)
        t=torch.randint(0,m.T,(x0.size(0),),device=device); xt,im=m.q_sample(x0,t)
        logits=m(xt,t,props,pmask,pad_mask=(xt==PAD))     # condition on all AVAILABLE props
        real=im&(x0!=PAD)&(x0!=BOS)&(x0!=EOS)
        if real.any():
            ce+=F.cross_entropy(logits[real].float(),x0[real],reduction='sum').item()
            acc+=(logits[real].argmax(-1)==x0[real]).sum().item(); cn+=real.sum().item()
        em=im&(x0==EOS)
        if em.any(): ec+=(logits[em].argmax(-1)==x0[em]).sum().item(); en+=em.sum().item()
    m.train(); return ce/max(cn,1), acc/max(cn,1), ec/max(en,1)

def _train_epoch(max_steps=None):
    dit9.train(); step=0
    for b in train_loader2:
        x0=b['tokens'].to(device); B=x0.size(0)
        props=b['props'].to(device); pmask=b['pmask'].to(device)
        keep=(torch.rand(B,9,device=device)>PROP_DROP).float()*pmask        # can't keep a missing prop
        allnull=(torch.rand(B,device=device)<ALLNULL_PROB).float().unsqueeze(1)
        keep=keep*(1-allnull)                                                # ~10% full-null steps
        t=torch.randint(0,dit9.T,(B,),device=device); xt,im=dit9.q_sample(x0,t)
        opt9.zero_grad()
        with torch.amp.autocast('cuda',dtype=torch.bfloat16,enabled=amp):
            logits=dit9(xt,t,props,keep,pad_mask=(xt==PAD))
            if not im.any(): continue
            loss=F.cross_entropy(logits[im].float(),x0[im])
        loss.backward(); torch.nn.utils.clip_grad_norm_(dit9.parameters(),1.0); opt9.step()
        step+=1
        if max_steps and step>=max_steps: break
    return step

if FAST_LOOP:
    print("=== GATE A: fast-loop ===")
    t0=time.time(); n=_train_epoch(max_steps=FAST_STEPS)
    vce,vacc,veos=_val9(dit9)
    print(f"after {n} steps ({time.time()-t0:.0f}s): "
          f"val_realCE={vce:.3f}  ppl={math.exp(vce):.1f}  acc={vacc:.3f}  eos={veos:.3f}")
    print("PASS if val_realCE trends toward ~1.3 and eos rising.")
    print("If PASS -> set FAST_LOOP=False and re-run this cell for the full train.")
else:
    wb=init_wandb("9prop_perpropdrop_d512_l6",
                  {'epochs':EPOCHS_S2,'lr':LR_S2,'prop_drop':PROP_DROP,
                   'allnull':ALLNULL_PROB,'n_props':9})
    best=float('inf'); history=[]
    def _payload(ep):
        return {'model':dit9.state_dict(),'opt':opt9.state_dict(),'epoch':ep,'best':best,
                'history':history,'n_props':9,'props':Stage2Dataset.PROPS}
    print(f"{'ep':>3}|{'val_realCE':>10}|{'ppl':>5}|{'acc':>5}|{'eos':>5}|{'sec':>4}")
    for ep in range(1,EPOCHS_S2+1):
        t0=time.time(); _train_epoch()
        vce,vacc,veos=_val9(dit9); sec=time.time()-t0
        rec={'epoch':ep,'val_realCE':vce,'ppl':math.exp(vce),'acc':vacc,'eos_acc':veos,'sec':sec}
        history.append(rec)
        print(f"{ep:>3}|{vce:>10.3f}|{math.exp(vce):>5.1f}|{vacc:>5.3f}|{veos:>5.3f}|{sec:>4.0f}")
        torch.save(_payload(ep), S2_CKPT/'latest.pt')
        if ep%10==0: torch.save(_payload(ep), S2_CKPT/f'epoch_{ep:03d}.pt')
        if vce<best:
            best=vce; torch.save(_payload(ep), S2_CKPT/'best.pt'); print(f"   best {vce:.3f} saved")
        json.dump(history, open(S2_DIR/'history.json','w'), indent=1)
        if wb: wb.log(rec)
    if wb: wb.finish()
    print("Phase 2 training complete ->", S2_CKPT)


In [ ]:
# ============================================================================
# CELL 25 · PHASE 2 (4/4) — props_to_cond + steering sweep + Gate B
#   Gate B uses an RDKit-MEASURABLE joint target {qed, mw, sa_score} -- proves
#   multi-objective control the QED-only model structurally cannot express.
#   (hERG / pchembl are trained into the model but need a surrogate scorer to
#    evaluate on novel molecules -> deferred; see plan.)
# ============================================================================
import json, numpy as np
from rdkit import Chem
from rdkit.Chem import QED, Descriptors

S2_NORM = json.load(open(config.model_dir/'stage2_norm_stats.json'))
PROPS9  = Stage2Dataset.PROPS

# optional SA scorer (RDKit Contrib); degrade gracefully if unavailable
try:
    from rdkit.Chem import RDConfig; import os, sys
    sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
    import sascorer
    _sa = lambda m: sascorer.calculateScore(m)
except Exception:
    _sa = None
    print("note: sascorer unavailable -> SA not scored in Gate B")

def props_to_cond(targets: dict):
    """targets e.g. {'qed':0.8,'mw':350}. Returns (props9, cmask9) on device;
       unspecified props -> null (cmask 0)."""
    P=np.zeros(len(PROPS9),dtype=np.float32); M=np.zeros(len(PROPS9),dtype=np.float32)
    for j,c in enumerate(PROPS9):
        if c in targets and targets[c] is not None:
            mu,sd=S2_NORM[c]['mean'],S2_NORM[c]['std']
            P[j]=(float(targets[c])-mu)/sd; M[j]=1.0
    return torch.tensor(P,device=device), torch.tensor(M,device=device)

@torch.no_grad()
def gen_mols(model, targets, n=256, cfg=1.0, temp=0.4, noise=0.0, steps=50):
    props,cmask=props_to_cond(targets)
    tk=sample(model,props,n=n,steps=steps,cfg=cfg,temperature=temp,noise=noise,cmask=cmask)
    out=[]
    for row in tk:
        s=tokenizer.decode(row.cpu().tolist())
        m=Chem.MolFromSmiles(s) if s else None
        if m is not None: out.append(m)
    return out

def _stats(mols):
    if not mols: return {}
    q =[QED.qed(m) for m in mols]; mw=[Descriptors.MolWt(m) for m in mols]
    d ={'n':len(mols),'meanQED':float(np.mean(q)),'maxQED':float(np.max(q)),
        'meanMW':float(np.mean(mw))}
    if _sa: d['meanSA']=float(np.mean([_sa(m) for m in mols]))
    return d

# ---- per-property QED steering sweep ----
print("=== 9-prop native QED steering (cfg=1, temp=0.4, noise=0) ===")
for tq in [0.3,0.5,0.7,0.9]:
    g=gen_mols(dit9,{'qed':tq},n=128)
    print(f"  target qed={tq}: valid={len(g)/128:.2f}  "
          f"meanQED={np.mean([QED.qed(m) for m in g]):.3f}  maxQED={np.max([QED.qed(m) for m in g]):.3f}")

# ---- GATE B: RDKit-measurable joint target ----
print("\n=== GATE B: joint target {qed:0.8, mw:350, sa_score:2.5}  (n=512) ===")
JT={'qed':0.8,'mw':350.0,'sa_score':2.5}
g9=gen_mols(dit9, JT, n=512)
print("  9-prop:", _stats(g9))
# baseline: QED-only model can only aim at qed; mw/sa are whatever falls out
try:
    gQ=gen_mols(dit_qedonly,{'qed':0.8},n=512) if 'dit_qedonly' in dir() else None
except Exception:
    gQ=None
if gQ is not None:
    print("  QED-only:", _stats(gQ))
print("\nPASS if the 9-prop joint-target generations sit closer to the FULL target "
      "(qed~0.8, mw~350, sa~2.5 together) than the QED-only model can.")


## Archive — one-off diagnostics & completed scripts
<sub>NOT part of the main run order. Stage-1 sweeps, the ablation generator (done → `notebooks/generate_stage1_ablation.py`), and one-off checkpoint / JSON patches. Kept for provenance only.</sub>

In [ ]:
# ARCHIVE 1 — sampler temperature × noise sweep (Stage-1 diagnostic, no retrain)
# ── SAMPLER TEMPERATURE × NOISE SWEEP  (no retraining)
# Tests whether low baseline QED is a sampler artifact (oversized molecules).
# Reports meanQED AND meanHA at low+high targets so we see baseline shift + steering spread.
from rdkit import Chem
from rdkit.Chem import QED
import numpy as np, pandas as pd

CFG       = 1.0
TARGETS   = [0.3, 0.9]
TEMPS     = [0.4, 0.6, 0.8, 1.0]
NOISES    = [0.0, 0.5]
N_PER     = 96
STEPS     = 50
DATA_HA   = 24.1   # training-set mean heavy atoms (target size)
DATA_QED  = 0.657  # training-set mean QED

def stats(tk_batch):
    qs, has = [], []
    for tk in tk_batch:
        mol = Chem.MolFromSmiles(tokenizer.decode(tk.cpu().tolist()))
        if mol:
            try: qs.append(QED.qed(mol)); has.append(mol.GetNumHeavyAtoms())
            except: pass
    n = len(qs)
    return (np.mean(qs) if qs else 0.0, np.mean(has) if has else 0.0, n)

print(f"data baseline: meanQED={DATA_QED:.3f}  meanHA={DATA_HA:.1f}\n")
print(f"{'temp':>4} {'noise':>5} | {'QED@0.3':>8} {'HA@0.3':>7} | {'QED@0.9':>8} {'HA@0.9':>7} | {'spread':>6} {'valid%':>6}")
print("-"*70)
rows = []
for temp in TEMPS:
    for noise in NOISES:
        res = {}
        for tgt in TARGETS:
            q, ha, nv = stats(sample(dit, qed_to_cond(tgt), n=N_PER, cfg=CFG,
                                     steps=STEPS, temperature=temp, noise=noise))
            res[tgt] = (q, ha, nv)
        q3, ha3, n3 = res[0.3]; q9, ha9, n9 = res[0.9]
        spread = q9 - q3; validpct = 100*(n3+n9)/(2*N_PER)
        print(f"{temp:>4.1f} {noise:>5.1f} | {q3:>8.3f} {ha3:>7.1f} | {q9:>8.3f} {ha9:>7.1f} | {spread:>+6.3f} {validpct:>5.0f}%")
        rows.append(dict(temp=temp, noise=noise, qed_lo=q3, ha_lo=ha3, qed_hi=q9, ha_hi=ha9, spread=spread))

print("\nWHAT TO LOOK FOR:")
print("  • meanHA dropping toward ~24 (from ~32) = molecules sized like the data")
print("  • QED@0.9 rising toward ~0.6 = baseline lifts when molecules aren't oversized")
print("  • spread growing = conditioning has more headroom once size is fixed")
df = pd.DataFrame(rows)


In [ ]:
# ARCHIVE 2 — Stage-1 final steering confirm at best operating point
# ── STAGE 1 FINAL: confirm steering at best operating point (temp=0.4, noise=0.0)
from rdkit import Chem
from rdkit.Chem import QED
import numpy as np, json

CFG, TEMP, NOISE, STEPS, N = 1.0, 0.4, 0.0, 50, 256
print(f"operating point: cfg={CFG} temp={TEMP} noise={NOISE}  (data: QED 0.657, HA 24.1)\n")
print(f"{'target':>7} | {'meanQED':>8} | {'meanHA':>7} | {'>0.7':>6} | {'valid':>6}")
print("-"*52)
rows=[]
for tgt in [0.3, 0.5, 0.7, 0.9]:
    qs, has = [], []
    for tk in sample(dit, qed_to_cond(tgt), n=N, cfg=CFG, steps=STEPS, temperature=TEMP, noise=NOISE):
        mol = Chem.MolFromSmiles(tokenizer.decode(tk.cpu().tolist()))
        if mol:
            try: qs.append(QED.qed(mol)); has.append(mol.GetNumHeavyAtoms())
            except: pass
    q = np.array(qs); frac = float((q>0.7).mean()) if len(q) else 0
    print(f"{tgt:>7.2f} | {q.mean():>8.3f} | {np.mean(has):>7.1f} | {frac:>5.1%} | {len(qs):>4}/{N}")
    rows.append(dict(target=tgt, meanQED=float(q.mean()), meanHA=float(np.mean(has)), frac_gt07=frac, valid=len(qs)))
json.dump(rows, open(RESULTS_DIR/'stage1_steering_final.json','w'), indent=1)
print("\n💾 saved → stage1_steering_final.json")


In [ ]:
# ARCHIVE 3 — CFG sweep (Stage-1 diagnostic, no retrain)
# ── CFG SWEEP  (no retraining)
import pandas as pd
from rdkit import Chem
from rdkit.Chem import QED
import numpy as np

TARGETS  = [0.3, 0.5, 0.7, 0.9]
CFG_VALS = [0.0, 0.5, 1.0, 2.0, 3.0, 4.0]
N_PER    = 64
STEPS    = 50

rows = []
for cfg_v in CFG_VALS:
    qeds_by_target = []
    for tgt in TARGETS:
        cond = qed_to_cond(tgt)
        qs = []
        for tk in sample(dit, cond, n=N_PER, cfg=cfg_v, steps=STEPS):
            mol = Chem.MolFromSmiles(tokenizer.decode(tk.cpu().tolist()))
            if mol:
                try: qs.append(QED.qed(mol))
                except: pass
        mean_q = float(np.mean(qs)) if qs else 0.0
        qeds_by_target.append(round(mean_q, 3))
        rows.append(dict(cfg=cfg_v, target=tgt, meanQED=mean_q, n_valid=len(qs)))
    spread = max(qeds_by_target) - min(qeds_by_target)
    print(f"cfg={cfg_v:.1f} | " + "  ".join(f"t{t}→{q:.3f}" for t,q in zip(TARGETS, qeds_by_target))
          + f"  [spread={spread:.3f}]")

print("\n--- pivot table ---")
df = pd.DataFrame(rows)
pivot = df.pivot(index='cfg', columns='target', values='meanQED')
print(pivot.to_string())


In [ ]:
# ARCHIVE 4 — Stage-1 ablation generator — DONE (canonical copy: notebooks/generate_stage1_ablation.py)
# ============================================================================
# STAGE 1 ABLATION — QED-only (1-prop) vs Multi-property (11-prop)
# Fully reproducible, reviewer-grade package: matched sampler, fixed seed, both
# models' full params, every number behind every figure, side-by-side comparison,
# best-of-N, QC, and environment capture.
#
# RUN INSIDE dit_conditional_generation.ipynb AFTER cells 1-15 + 17, so these
# globals exist: tokenizer, sample, device, config, QMEAN, QSTD, PAD, BOS, EOS,
# MASKi, train_df, train_dataset, and DiTBlock / TimestepEmbed / modulate (cell 14).
#
# Nothing is fabricated — every number comes from real generation on your models.
# ============================================================================
import os, sys, json, math, time, platform, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from rdkit import Chem, rdBase
from rdkit.Chem import QED, RDConfig, AllChem, DataStructs

# ---- CONFIG -----------------------------------------------------------------
QEDONLY_CKPT = CKPT_DIR / 'best.pt'                       # 1-prop, wide MLP
MULTI_CKPT   = config.model_dir / 'cond_dit_best.pt'      # 11-prop, narrow MLP
ABL          = config.model_dir / 'stage1_ablation'

SEED      = 42                                             # <-- makes the whole run reproducible
CFG, TEMP, NOISE, STEPS = 1.0, 0.4, 0.0, 50               # matched operating point
TARGETS   = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
N_STEER   = 256
N_BESTN   = 512
KEEP_BESTN= 50
BESTN_TGT = 0.9
ASSUMED_HEADS = 8                                          # nh isn't stored in weights; cell-14 default

# Training settings NOT stored in the checkpoint — documented from project history.
# (epoch IS read authoritatively from each checkpoint; these are the rest.)
TRAIN_SETTINGS = {
    'QED-only':   dict(cfg_dropout=0.3, lr=3e-4, batch=64,
                       note='wide property-MLP retrain (this session)'),
    'Multi-prop': dict(cfg_dropout=0.1, lr=3e-4, batch=64,
                       note='original narrow property-MLP model'),
}

for sub in ('csv', 'figures', 'tables', 'config', 'figure_data'):
    (ABL / sub).mkdir(parents=True, exist_ok=True)
print(f"📁 ablation output → {ABL}")

# ---- reproducibility seed ---------------------------------------------------
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# ---- SA score + novelty reference ------------------------------------------
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
import sascorer
_train_canon = set()
for s in train_df['canonical_smiles'].tolist():
    m = Chem.MolFromSmiles(str(s))
    if m: _train_canon.add(Chem.MolToSmiles(m))
print(f"   novelty reference: {len(_train_canon):,} training molecules")

# ---- Flexible CondDiT (reuses cell-14 DiTBlock/TimestepEmbed/modulate) -------
class FlexCondDiT(nn.Module):
    def __init__(self, V, max_len, n_props, d=512, nh=8, nl=6, T=100, dr=0.1,
                 pm_mult=4, pad_id=0, bos_id=1, eos_id=2, mask_id=3):
        super().__init__()
        self.tok = nn.Embedding(V, d); self.pos = nn.Embedding(max_len, d); self.te = TimestepEmbed(d)
        self.pm = nn.Sequential(nn.Linear(n_props, pm_mult*d), nn.SiLU(), nn.Linear(pm_mult*d, d))
        self.null = nn.Parameter(torch.zeros(n_props))
        self.blocks = nn.ModuleList([DiTBlock(d, nh, dr) for _ in range(nl)])
        self.nf = nn.LayerNorm(d, elementwise_affine=False, eps=1e-6)
        self.af = nn.Sequential(nn.SiLU(), nn.Linear(d, 2*d))
        self.head = nn.Linear(d, V)
        self.T=T; self.max_len=max_len; self.pad_id=pad_id; self.bos_id=bos_id
        self.eos_id=eos_id; self.mask_id=mask_id; self.vocab_size=V; self.n_props=n_props
        s = torch.arange(T+1)/T; ab = torch.cos((s+0.008)/1.008 * math.pi/2)**2
        self.register_buffer('alpha_bars', (ab/ab[0])[1:])
    def forward(self, xt, t, props, use_cond, pad_mask):
        B, L = xt.shape
        pos = torch.arange(L, device=xt.device).unsqueeze(0).expand(B, -1)
        h = self.tok(xt) + self.pos(pos)
        uc = use_cond.view(B, 1).float()
        p = uc * props + (1 - uc) * self.null.unsqueeze(0)
        c = self.te(t) + self.pm(p)
        for blk in self.blocks: h = blk(h, c, pad_mask)
        sh, sc = self.af(c).chunk(2, -1); h = modulate(self.nf(h), sh, sc)
        return self.head(h)

def load_any(path, tag):
    ck = torch.load(path, map_location=device)
    sd = ck['model'] if 'model' in ck else ck
    d       = sd['tok.weight'].shape[1]
    V       = sd['tok.weight'].shape[0]
    max_len = sd['pos.weight'].shape[0]
    n_props = sd['null'].shape[0]
    pm_mult = sd['pm.0.weight'].shape[0] // d
    nl      = len({k.split('.')[1] for k in sd if k.startswith('blocks.')})
    T       = sd['alpha_bars'].shape[0] if 'alpha_bars' in sd else 100
    m = FlexCondDiT(V, max_len, n_props, d=d, nh=ASSUMED_HEADS, nl=nl, T=T, pm_mult=pm_mult,
                    pad_id=PAD, bos_id=BOS, eos_id=EOS, mask_id=MASKi).to(device)
    m.load_state_dict(sd); m.eval()
    n_param = sum(p.numel() for p in m.parameters())
    # full checkpoint metadata (everything except the big state_dict tensors)
    ck_meta = {k: (str(v) if not torch.is_tensor(v) else f'<tensor {tuple(v.shape)}>')
               for k, v in ck.items() if k != 'model'}
    params = dict(tag=tag, path=str(path), vocab_size=V, d_model=d, n_layers=nl,
                  n_heads=ASSUMED_HEADS, n_props=n_props, pm_mult=pm_mult, T=T,
                  max_len=max_len, n_parameters=n_param,
                  checkpoint_epoch=ck.get('epoch', '?'), checkpoint_best_val=ck.get('best', '?'),
                  checkpoint_keys=list(ck.keys()), checkpoint_meta=ck_meta,
                  file_size_mb=round(Path(path).stat().st_size/1e6, 2),
                  **TRAIN_SETTINGS.get(tag, {}))
    print(f"   ✅ {tag}: n_props={n_props} d={d} nl={nl} pm_mult={pm_mult} params={n_param:,} "
          f"(epoch {params['checkpoint_epoch']}, val {params['checkpoint_best_val']})")
    return m, params

# ---- conditioning: QED at idx 0, other props held at dataset mean (z=0) ------
def cond_vec(model, target_qed):
    v = torch.zeros(model.n_props, device=device)
    v[0] = (target_qed - QMEAN) / QSTD
    return v

def decode_mol(tk):
    m = Chem.MolFromSmiles(tokenizer.decode(tk.cpu().tolist()))
    if m is None: return None
    try: return dict(qed=QED.qed(m), ha=m.GetNumHeavyAtoms(), smiles=Chem.MolToSmiles(m))
    except Exception: return None

@torch.no_grad()
def steering(model, tag):
    summ, permol = [], []
    for tq in TARGETS:
        recs = [r for r in (decode_mol(tk) for tk in
                sample(model, cond_vec(model, tq), n=N_STEER, cfg=CFG, steps=STEPS,
                       temperature=TEMP, noise=NOISE)) if r]
        q  = np.array([r['qed'] for r in recs]) if recs else np.array([0.0])
        ha = np.array([r['ha']  for r in recs]) if recs else np.array([0.0])
        uniq = len({r['smiles'] for r in recs}) / max(len(recs), 1)
        summ.append(dict(model=tag, target=tq, n_valid=len(recs),
                         meanQED=float(q.mean()), stdQED=float(q.std()), medianQED=float(np.median(q)),
                         meanHA=float(ha.mean()), frac_gt07=float((q > 0.7).mean()), uniqueness=uniq))
        for r in recs: permol.append(dict(model=tag, target=tq, **r))
        print(f"      {tag:10s} t{tq:.1f}: meanQED={q.mean():.3f} meanHA={ha.mean():.1f} "
              f">0.7={100*(q>0.7).mean():.1f}% valid={len(recs)}")
    return pd.DataFrame(summ), pd.DataFrame(permol)

@torch.no_grad()
def best_of_n(model, tag):
    recs = [r for r in (decode_mol(tk) for tk in
            sample(model, cond_vec(model, BESTN_TGT), n=N_BESTN, cfg=CFG, steps=STEPS,
                   temperature=TEMP, noise=NOISE)) if r]
    df = pd.DataFrame(recs).sort_values('qed', ascending=False).reset_index(drop=True)
    df.insert(0, 'model', tag)
    print(f"      {tag:10s} best-of-{N_BESTN}@{BESTN_TGT}: best={df['qed'].iloc[0]:.3f} "
          f"median(top{KEEP_BESTN})={df['qed'].head(KEEP_BESTN).median():.3f}")
    return df

def internal_diversity(smiles, cap=200):
    fps = [AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048)
           for m in (Chem.MolFromSmiles(s) for s in smiles[:cap]) if m]
    if len(fps) < 2: return 0.0
    sims = []
    for i in range(len(fps)): sims += list(DataStructs.BulkTanimotoSimilarity(fps[i], fps[i+1:]))
    return float(1 - np.mean(sims)) if sims else 0.0

def qc_row(tag, bestn_df, permol):
    smis = permol[permol.model == tag]['smiles'].tolist()
    canon = [s for s in smis if Chem.MolFromSmiles(s)]
    novel = [s for s in canon if s not in _train_canon]
    sa = []
    for s in canon[:500]:
        m = Chem.MolFromSmiles(s)
        if m:
            try: sa.append(sascorer.calculateScore(m))
            except Exception: pass
    return dict(model=tag, n=len(smis), validity=len(canon)/max(len(smis),1),
                uniqueness=len(set(canon))/max(len(canon),1), novelty=len(novel)/max(len(canon),1),
                mean_SA=float(np.mean(sa)) if sa else float('nan'),
                frac_SA_lt45=float(np.mean([x<4.5 for x in sa])) if sa else float('nan'),
                internal_diversity=internal_diversity(canon),
                bestN_top_qed=float(bestn_df['qed'].iloc[0]),
                bestN_median_topK=float(bestn_df['qed'].head(KEEP_BESTN).median()))

# ============================================================================
# RUN
# ============================================================================
t0 = time.time()
print("\n[1/5] loading models…")
models, params = {}, {}
m1, p1 = load_any(QEDONLY_CKPT, 'QED-only'); models['QED-only'] = m1; params['QED-only'] = p1
if Path(MULTI_CKPT).exists():
    m11, p11 = load_any(MULTI_CKPT, 'Multi-prop'); models['Multi-prop'] = m11; params['Multi-prop'] = p11
else:
    print(f"   ⚠️ multi-prop checkpoint not found at {MULTI_CKPT} — running QED-only only.")

print("\n[2/5] native steering…")
ss, sp = [], []
for tag, m in models.items():
    s, pm = steering(m, tag); ss.append(s); sp.append(pm)
steer_summ = pd.concat(ss, ignore_index=True); steer_permol = pd.concat(sp, ignore_index=True)

print("\n[3/5] best-of-N…")
bestn = {tag: best_of_n(m, tag) for tag, m in models.items()}

print("\n[4/5] QC…")
qc = pd.DataFrame([qc_row(tag, bestn[tag], steer_permol) for tag in models])

print("\n[5/5] comparison + saving everything…")
tag2slug = {'QED-only': 'qedonly', 'Multi-prop': 'multiprop'}

# ---- per-model CSVs ---------------------------------------------------------
for tag in models:
    slug = tag2slug[tag]
    steer_summ[steer_summ.model==tag].to_csv(ABL/'csv'/f'native_steering_{slug}.csv', index=False)
    steer_permol[steer_permol.model==tag].to_csv(ABL/'csv'/f'native_steering_permol_{slug}.csv', index=False)
    bestn[tag].to_csv(ABL/'csv'/f'bestofN_{slug}_qed{BESTN_TGT}.csv', index=False)
qc.to_csv(ABL/'csv'/'qc_summary.csv', index=False)

# ---- side-by-side COMPARISON (the headline deliverable) ---------------------
comparison = None
if len(models) == 2:
    a = steer_summ[steer_summ.model=='QED-only'].set_index('target')
    b = steer_summ[steer_summ.model=='Multi-prop'].set_index('target')
    comparison = pd.DataFrame({
        'target': a.index,
        'QEDonly_meanQED':  a['meanQED'].values,
        'Multiprop_meanQED':b['meanQED'].values,
        'delta_meanQED':    (b['meanQED']-a['meanQED']).values,
        'QEDonly_meanHA':   a['meanHA'].values,
        'Multiprop_meanHA': b['meanHA'].values,
        'QEDonly_frac>0.7': a['frac_gt07'].values,
        'Multiprop_frac>0.7':b['frac_gt07'].values,
    })
    comparison.to_csv(ABL/'csv'/'comparison_by_target.csv', index=False)
    # headline summary (spread + best-of-N) per model
    head = []
    for tag in models:
        d = steer_summ[steer_summ.model==tag]
        head.append(dict(model=tag,
                         steering_spread=float(d.meanQED.max()-d.meanQED.min()),
                         max_meanQED=float(d.meanQED.max()),
                         bestN_top=float(bestn[tag]['qed'].iloc[0]),
                         bestN_median_topK=float(bestn[tag]['qed'].head(KEEP_BESTN).median())))
    head_df = pd.DataFrame(head)
    head_df.to_csv(ABL/'tables'/'headline_comparison.csv', index=False)
    with open(ABL/'tables'/'comparison_table.md', 'w') as f:
        f.write("## Native steering — side by side\n\n")
        f.write(comparison.round(3).to_markdown(index=False))
        f.write("\n\n## Headline comparison\n\n")
        f.write(head_df.round(3).to_markdown(index=False))

steer_summ.to_csv(ABL/'tables'/'summary_table.csv', index=False)
with open(ABL/'tables'/'summary_table.md', 'w') as f:
    f.write(steer_summ.round(3).to_markdown(index=False))

# ---- per-figure backing data (every number behind every graph) --------------
steer_summ.pivot(index='target', columns='model', values=['meanQED','stdQED']).to_csv(ABL/'figure_data'/'fig1_meanQED_vs_target.csv')
steer_summ.pivot(index='target', columns='model', values='meanHA').to_csv(ABL/'figure_data'/'fig2_meanHA_vs_target.csv')
steer_summ.pivot(index='target', columns='model', values='frac_gt07').to_csv(ABL/'figure_data'/'fig3_frac_gt07.csv')
pd.concat([bestn[t][['model','qed']] for t in models]).to_csv(ABL/'figure_data'/'fig4_bestofN_qed.csv', index=False)
steer_permol[['model','target','qed']].to_csv(ABL/'figure_data'/'fig5_qed_by_target.csv', index=False)

# ---- FIGURES ----------------------------------------------------------------
COL = {'QED-only':'#d62728', 'Multi-prop':'#1f77b4'}
plt.figure(figsize=(6,4.2))
for tag in models:
    d = steer_summ[steer_summ.model==tag]
    plt.errorbar(d.target, d.meanQED, yerr=d.stdQED, marker='o', capsize=3, label=tag, color=COL[tag])
plt.plot([0.3,0.9],[0.3,0.9],'k--',alpha=.4,label='ideal (y=x)')
plt.axhline(0.657,color='gray',ls=':',alpha=.6,label='data mean QED')
plt.xlabel('target QED'); plt.ylabel('achieved meanQED'); plt.title('Native QED steering')
plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(ABL/'figures'/'fig1_native_steering_meanQED.png', dpi=300); plt.close()

plt.figure(figsize=(6,4.2))
for tag in models:
    d = steer_summ[steer_summ.model==tag]
    plt.plot(d.target, d.meanHA, marker='s', label=tag, color=COL[tag])
plt.axhline(24.1,color='gray',ls=':',alpha=.6,label='data mean HA')
plt.xlabel('target QED'); plt.ylabel('mean heavy atoms'); plt.title('Molecule size vs QED target')
plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(ABL/'figures'/'fig2_size_vs_target_meanHA.png', dpi=300); plt.close()

plt.figure(figsize=(6,4.2)); w=0.35; x=np.arange(len(TARGETS))
for i,tag in enumerate(models):
    d = steer_summ[steer_summ.model==tag].sort_values('target')
    plt.bar(x+(i-0.5)*w, d.frac_gt07.values, w, label=tag, color=COL[tag])
plt.xticks(x,[f'{t:.1f}' for t in TARGETS]); plt.xlabel('target QED'); plt.ylabel('fraction QED > 0.7')
plt.title('High-QED hit rate'); plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(ABL/'figures'/'fig3_frac_above_0.7.png', dpi=300); plt.close()

plt.figure(figsize=(6,4.2))
for tag in models: plt.hist(bestn[tag]['qed'], bins=30, alpha=.5, label=tag, color=COL[tag])
plt.axvline(0.657,color='gray',ls=':',alpha=.6,label='data mean')
plt.xlabel('QED'); plt.ylabel('count'); plt.title(f'Best-of-{N_BESTN} pool @ target {BESTN_TGT}')
plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(ABL/'figures'/'fig4_bestofN_distribution.png', dpi=300); plt.close()

plt.figure(figsize=(7,4.2)); tags=list(models); data,positions,colors=[],[],[]
for ti,tq in enumerate(TARGETS):
    for mi,tag in enumerate(tags):
        data.append(steer_permol[(steer_permol.model==tag)&(steer_permol.target==tq)]['qed'].values)
        colors.append(COL[tag]); positions.append(ti*(len(tags)+1)+mi)
bp = plt.boxplot(data, positions=positions, widths=.8, patch_artist=True, showfliers=False)
for patch,c in zip(bp['boxes'],colors): patch.set_facecolor(c); patch.set_alpha(.6)
plt.xticks([ti*(len(tags)+1)+0.5 for ti in range(len(TARGETS))],[f'{t:.1f}' for t in TARGETS])
plt.xlabel('target QED'); plt.ylabel('QED'); plt.title('QED distribution by target')
plt.legend([Patch(facecolor=COL[t],alpha=.6) for t in tags], tags, fontsize=8)
plt.tight_layout(); plt.savefig(ABL/'figures'/'fig5_qed_boxplot_by_target.png', dpi=300); plt.close()

# ---- CONFIG / PARAMS / ENVIRONMENT (full reproducibility) -------------------
json.dump(params, open(ABL/'config'/'model_params.json','w'), indent=2, default=str)

run_config = dict(seed=SEED, sampler=dict(cfg=CFG, temperature=TEMP, noise=NOISE, steps=STEPS),
                  targets=TARGETS, n_steer=N_STEER, n_bestN=N_BESTN, keep_bestN=KEEP_BESTN,
                  bestN_target=BESTN_TGT,
                  conditioning='QED at index 0 (z-scored); for 11-prop the other 10 held at dataset mean (z=0)',
                  qed_norm=dict(mean=QMEAN, std=QSTD),
                  qed_zscore_per_target={str(t): (t-QMEAN)/QSTD for t in TARGETS})
json.dump(run_config, open(ABL/'config'/'run_config.json','w'), indent=2, default=str)

# full norm_stats (documents exactly how every property was normalized)
try: json.dump(train_dataset.norm_stats, open(ABL/'config'/'norm_stats.json','w'), indent=2, default=str)
except Exception as e: print("   (norm_stats not saved:", e, ")")

def _git():
    try: return subprocess.check_output(['git','rev-parse','HEAD'], stderr=subprocess.DEVNULL).decode().strip()
    except Exception: return None
env = dict(python=sys.version.split()[0], platform=platform.platform(),
           torch=torch.__version__, numpy=np.__version__, pandas=pd.__version__,
           rdkit=rdBase.rdkitVersion, cuda=torch.version.cuda,
           gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
           git_commit=_git())
json.dump(env, open(ABL/'config'/'environment.json','w'), indent=2, default=str)

manifest = dict(run_config=run_config, environment=env,
                models={t: {k: params[t][k] for k in
                            ('n_props','d_model','n_layers','n_heads','pm_mult','T','max_len',
                             'n_parameters','checkpoint_epoch','checkpoint_best_val','cfg_dropout')}
                        for t in models},
                qc=qc.to_dict('records'), runtime_sec=round(time.time()-t0,1),
                caveat="As-trained comparison: checkpoints differ in epochs/MLP-width/cfg-dropout. "
                       "Qualitative conclusion robust; retrain matched for a bulletproof claim.")
json.dump(manifest, open(ABL/'manifest.json','w'), indent=2, default=str)

# ---- console summary --------------------------------------------------------
print("\n================  MODEL PARAMS  ================")
for tag in models:
    p = params[tag]
    print(f"  {tag:10s}: n_props={p['n_props']} d={p['d_model']} layers={p['n_layers']} "
          f"heads={p['n_heads']} pm_mult={p['pm_mult']} params={p['n_parameters']:,} "
          f"epoch={p['checkpoint_epoch']} cfg_dropout={p.get('cfg_dropout','?')}")
print("\n================  QC SUMMARY  ==================")
print(qc.round(3).to_string(index=False))
if comparison is not None:
    print("\n================  COMPARISON (native steering)  ================")
    print(comparison.round(3).to_string(index=False))
    print("\n  headline:"); print(head_df.round(3).to_string(index=False))
print(f"\n✅ done in {manifest['runtime_sec']}s — everything under {ABL}")
print("   download that folder into  notebooks/results/stage1_ablation/  for the paper.")


In [ ]:
# ARCHIVE 5 — one-off: inspect cond_dit_best.pt checkpoint payload
ck = torch.load(config.model_dir/'cond_dit_best.pt', map_location='cpu')
print("type:", type(ck))
print("top-level keys:", list(ck.keys()) if isinstance(ck, dict) else "raw state_dict (no metadata)")
for k, v in (ck.items() if isinstance(ck, dict) else []):
    if k != 'model':
        print(f"  {k} = {v}")


In [ ]:
# ARCHIVE 6 — one-off: model_params.json provenance patch — DONE
import json
mp_path = ABL/'config'/'model_params.json'
mp = json.load(open(mp_path))
mp['Multi-prop']['checkpoint_epoch']    = '~40 (labelled "ep40" in best_of_n_baseline CSV; NOT stored in checkpoint)'
mp['Multi-prop']['checkpoint_best_val'] = 'not stored'
mp['Multi-prop']['provenance_note'] = ('checkpoint payload = {model, n_props} only. Architecture inferred '
    'from weights (authoritative). Training settings (cfg_dropout=0.1, lr=3e-4, batch=64, narrow MLP, '
    '~epoch 40) documented from project history, not stored in the checkpoint.')
json.dump(mp, open(mp_path, 'w'), indent=2, default=str)
print("✅ patched", mp_path)
